In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 1.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import numpy as np

from pyproj import Transformer
from sklearn.neighbors import BallTree
from pathlib import Path
import requests
import zipfile
from pathlib import Path

In [4]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Movilidad_inteligente_madrid")

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
GRAPH_DIR = DATA_DIR / "graphs"
VELOCIDADES_DIR = BASE_DIR / "mapa_velocidades"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
VELOCIDADES_DIR.mkdir(parents=True, exist_ok=True)

ruta_velocidades = VELOCIDADES_DIR / "velocidades_madrid_final.geojson"

print("BASE_DIR:", BASE_DIR)
print("Existe GeoJSON velocidades:", ruta_velocidades.exists())
print("Ruta velocidades:", ruta_velocidades)

BASE_DIR: /content/drive/MyDrive/Movilidad_inteligente_madrid
Existe GeoJSON velocidades: True
Ruta velocidades: /content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson


In [5]:
from pathlib import Path
from google.colab import drive

# Montar Drive si todavía no está montado
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# Definir MYDRIVE
MYDRIVE = Path("/content/drive/MyDrive")

print("MYDRIVE:", MYDRIVE)
print("Existe:", MYDRIVE.exists())

MYDRIVE: /content/drive/MyDrive
Existe: True


In [6]:
from pathlib import Path
import zipfile

MYDRIVE = Path("/content/drive/MyDrive")

# zip descargado y se lee desde google drive
resultados_zip = list(MYDRIVE.rglob("202468-292-intensidad-trafico.zip"))
if len(resultados_zip) == 0:
       resultados_zip = [Path("/content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/202468-292-intensidad-trafico.zip")]

print("ZIP encontrados:", len(resultados_zip))

for r in resultados_zip:
    print(r)

if len(resultados_zip) == 0:
    raise FileNotFoundError("No se ha encontrado el ZIP en Google Drive.")

ruta_zip_medidores = resultados_zip[0]

#rutas
RAW_DIR = ruta_zip_medidores.parent
DATA_DIR = RAW_DIR.parent
BASE_DIR = DATA_DIR.parent
PROCESSED_DIR = DATA_DIR / "processed"
GRAPH_DIR = DATA_DIR / "graphs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

carpeta_medidores = RAW_DIR / "pmed_ubicacion"

# Crear carpeta de extracción
carpeta_medidores.mkdir(parents=True, exist_ok=True)

print("Ruta ZIP:", ruta_zip_medidores)
print("Carpeta extracción:", carpeta_medidores)

#contenido del zip
with zipfile.ZipFile(ruta_zip_medidores, "r") as zip_ref:
    print("Contenido del ZIP:")
    for nombre in zip_ref.namelist():
        print(nombre)

    # Extraer todo
    zip_ref.extractall(carpeta_medidores)

print("\nArchivos extraídos:")
for p in carpeta_medidores.iterdir():
    print(p.name)

# Buscar shapefile
shapefiles = list(carpeta_medidores.rglob("*.shp"))

print("\nShapefiles encontrados:", len(shapefiles))

for shp in shapefiles:
    print(shp)

if len(shapefiles) == 0:
    raise FileNotFoundError("No se ha encontrado ningún archivo .shp después de descomprimir.")

ruta_shp_medidores = shapefiles[0]

print("Usando shapefile:", ruta_shp_medidores)

ZIP encontrados: 1
/content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/202468-292-intensidad-trafico.zip
Ruta ZIP: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/202468-292-intensidad-trafico.zip
Carpeta extracción: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/pmed_ubicacion
Contenido del ZIP:
pmed_ubicacion_05-2026.cpg
pmed_ubicacion_05-2026.dbf
pmed_ubicacion_05-2026.prj
pmed_ubicacion_05-2026.shp
pmed_ubicacion_05-2026.shx

Archivos extraídos:
pmed_ubicacion_05-2026.cpg
pmed_ubicacion_05-2026.prj
pmed_ubicacion_05-2026.dbf
pmed_ubicacion_05-2026.shx
pmed_ubicacion_05-2026.shp

Shapefiles encontrados: 1
/content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/pmed_ubicacion/pmed_ubicacion_05-2026.shp
Usando shapefile: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/pmed_ubicacion/pmed_ubicacion_05-2026.shp


In [7]:
from pathlib import Path
ruta = Path("/content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson")
print("Existe:", ruta.exists())
print(list(ruta.parent.iterdir()))

Existe: True
[PosixPath('/content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson')]


In [8]:
resultados = list(MYDRIVE.rglob("velocidades_madrid_final.geojson"))

print("Archivos encontrados:", len(resultados))

for r in resultados:
    print(r)

Archivos encontrados: 1
/content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson


In [9]:
ruta_velocidades = Path("/content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson")
print("Ruta velocidades:", ruta_velocidades)
print("Existe:", ruta_velocidades.exists())

Ruta velocidades: /content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson
Existe: True


In [10]:
ruta_velocidades = Path("/content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson")

print("Ruta velocidades:", ruta_velocidades)
print("Existe:", ruta_velocidades.exists())

print("Ruta velocidades:", ruta_velocidades)
print("Existe:", ruta_velocidades.exists())

Ruta velocidades: /content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson
Existe: True
Ruta velocidades: /content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson
Existe: True


In [11]:
gdf_velocidades_full = gpd.read_file(ruta_velocidades)

print("Registros:", len(gdf_velocidades_full))
print("Columnas:")
print(gdf_velocidades_full.columns.tolist())

gdf_velocidades_full.head()

Registros: 271232
Columnas:
['u', 'v', 'key', 'highway', 'maxspeed', 'es_urbano', 'maxspeed_final', 'geometry']


,u,v,key,highway,maxspeed,es_urbano,maxspeed_final,geometry
0,21741584,21741587,0,[motorway],[100],False,[100],"LINESTRING (-3.69059 40.26517, -3.6905 40.2628..."
1,21741587,759985345,0,[motorway_link],"[40, 70]",False,"[40, 70]","LINESTRING (-3.69004 40.24808, -3.6901 40.2477..."
2,21741587,310025091,0,[motorway],[100],False,[100],"LINESTRING (-3.69004 40.24808, -3.68994 40.245..."
3,21741595,1317348460,0,"[motorway_link, tertiary]",[40],True,[40],"LINESTRING (-3.67595 40.1961, -3.6759 40.19589..."
4,21741595,310031145,0,[motorway],"[100, 120]",False,"[100, 120]","LINESTRING (-3.67595 40.1961, -3.67566 40.1956..."


In [12]:
print(ruta_shp_medidores)

/content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/pmed_ubicacion/pmed_ubicacion_05-2026.shp


In [13]:
gdf_medidores = gpd.read_file(ruta_shp_medidores)

print("Filas:", len(gdf_medidores))
print("Columnas:", gdf_medidores.columns.tolist())

gdf_medidores.head()

Filas: 5072
Columnas: ['TIPO_ELEM', 'DISTRITO', 'ID', 'COD_CENT', 'NOMBRE', 'UTM_X', 'UTM_Y', 'LONGITUD', 'LATITUD', 'geometry']


,TIPO_ELEM,DISTRITO,ID,COD_CENT,NOMBRE,UTM_X,UTM_Y,LONGITUD,LATITUD,geometry
0,other,1.0,6835,18RA28PM01,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315,"POLYGON ((438768.544 4474328.066, 438766.024 4..."
1,other,9.0,1012,18RA66PM01,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861,"POLYGON ((438738.316 4474613.231, 438738.979 4..."
2,URB,10.0,5035,95013,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040,"POLYGON ((438001.652 4473855.413, 438004.268 4..."
3,URB,5.0,5579,61068,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932,"POLYGON ((442417.854 4478699.064, 442418.653 4..."
4,URB,5.0,5580,61069,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073,"POLYGON ((442370.039 4478603.48, 442367.151 44..."


In [14]:
gdf_medidores.columns = gdf_medidores.columns.str.lower()

medidores = gdf_medidores[
    ["id", "nombre", "utm_x", "utm_y", "longitud", "latitud"]
].copy()

medidores = medidores.dropna(subset=["longitud", "latitud"])
medidores = medidores.drop_duplicates(subset=["id"])

medidores["latitud"] = medidores["latitud"].astype(float)
medidores["longitud"] = medidores["longitud"].astype(float)

print("Medidores:", len(medidores))
medidores.head()

Medidores: 5072


,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [15]:
medidores.dtypes

,0
id,int64
nombre,object
utm_x,float64
utm_y,float64
longitud,float64
latitud,float64


In [16]:
#calidad de datos
print("IDs duplicados:", medidores["id"].duplicated().sum())
print("Latitud nula:", medidores["latitud"].isna().sum())
print("Longitud nula:", medidores["longitud"].isna().sum())

fuera_madrid = medidores[
    ~(
        (medidores["latitud"].between(40.30, 40.55)) &
        (medidores["longitud"].between(-3.90, -3.50))
    )
]

print("Medidores fuera de rango Madrid:", len(fuera_madrid))
fuera_madrid.head()

IDs duplicados: 0
Latitud nula: 0
Longitud nula: 0
Medidores fuera de rango Madrid: 0


,id,nombre,utm_x,utm_y,longitud,latitud


In [17]:
medidores[["latitud", "longitud"]].describe()

,latitud,longitud
count,5072.000000,5072.000000
mean,40.430447,-3.684001
std,0.039163,0.042728
min,40.332454,-3.836886
25%,40.398978,-3.712553
50%,40.431302,-3.686923
75%,40.460080,-3.656194
max,40.515611,-3.551623


## Descarga de la red viaria de Madrid con OSMnx

In [18]:
ruta_grafo_osm_base = GRAPH_DIR / "red_osm_madrid_base.graphml"

if ruta_grafo_osm_base.exists():
    print("Cargando red OSM desde Drive")
    G_osm = ox.load_graphml(ruta_grafo_osm_base)
else:
    print("No existe red OSM. Descargando desde OSMnx...")
    G_osm = ox.graph_from_place(
        "Madrid, Spain",
        network_type="drive",
        simplify=True
    )
    ox.save_graphml(G_osm, filepath=ruta_grafo_osm_base)

print("Nodos OSM:", G_osm.number_of_nodes())
print("Aristas OSM:", G_osm.number_of_edges())
print("Ruta GraphML:", ruta_grafo_osm_base)

Cargando red OSM desde Drive
Nodos OSM: 31443
Aristas OSM: 61843
Ruta GraphML: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/graphs/red_osm_madrid_base.graphml


## Snap de medidores a la red OSM

Cada punto medidor se asocia al nodo OSM más cercano.  
 `snap to graph + camino más corto sobre OSMnx`.

In [19]:
osm_nodes, distancias = ox.distance.nearest_nodes(
    G_osm,
    X=medidores["longitud"].values,
    Y=medidores["latitud"].values,
    return_dist=True
)

medidores["osm_node"] = osm_nodes
medidores["distancia_osm_node_m"] = distancias

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315,32636471,79.621034
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861,315259372,54.829432
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040,305399713,15.640852
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932,1672792326,40.116809
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073,119794656,14.979371


In [20]:
medidores["distancia_osm_node_m"].describe()

,distancia_osm_node_m
count,5072.000000
mean,36.615560
std,28.020252
min,0.101869
25%,16.220982
50%,28.055324
75%,50.402014
max,345.242185


In [21]:
medidores.sort_values("distancia_osm_node_m", ascending=False).head(20)

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
1938,5268,(TACTICO)SALIDA POLIGONO N-S,434514.277528,4.468632e+06,-3.771300,40.365688,306400716,345.242185
107,4928,(TACTICO) AV. POBLADOS O-E (GIRO A ERICA),435668.604184,4.470451e+06,-3.757889,40.382164,282940163,228.695575
1394,4960,(TACTICO) ERICA N-S (CENTRO C.I.E.),435691.441140,4.470458e+06,-3.757621,40.382233,282940163,219.565346
1760,11199,Fuerzas Armadas - Ciudad Deportiva O-E - Fuerz...,448191.892131,4.481464e+06,-3.611259,40.482247,1012899036,178.720804
2325,11200,Fuerzas Armadas - Ciudad Deportiva O-E (Vía Se...,448192.910496,4.481435e+06,-3.611244,40.481993,1012899118,178.333987
4888,6876,12XC06PM01,441861.911399,4.471142e+06,-3.684994,40.388851,317771984,169.480943
1745,11191,"Av Fuerzas Armadas, 322 O-E - Av Fuerzas Armad...",447371.205461,4.481464e+06,-3.620941,40.482198,969169634,168.795375
1759,11192,"Av Fuerzas Armadas, 322 O-E (Via Servicio) - A...",447372.223825,4.481436e+06,-3.620927,40.481944,969169634,168.326941
446,9916,SINESIO DELGADO O-E (HOSPITAL CARLOS III-ENTRA...,440954.801598,4.480675e+06,-3.696566,40.474658,26205041,163.715082
445,9915,SINESIO DELGADO E-O (SALIDA TUNEL-HOSPITAL CAR...,440949.748176,4.480680e+06,-3.696627,40.474708,26205041,156.229910


## Umbral adaptativo por zona

In [22]:
# Cálculo de umbral adaptativo por zona


medidores = medidores.reset_index(drop=True)

gdf_medidores_proj = gpd.GeoDataFrame(
    medidores.copy(),
    geometry=gpd.points_from_xy(medidores["longitud"], medidores["latitud"]),
    crs="EPSG:4326"
).to_crs("EPSG:25830")

CELL_SIZE_M = 1000  #1km x 1km

gdf_medidores_proj["x"] = gdf_medidores_proj.geometry.x
gdf_medidores_proj["y"] = gdf_medidores_proj.geometry.y

gdf_medidores_proj["cell_x"] = (gdf_medidores_proj["x"] // CELL_SIZE_M).astype(int)
gdf_medidores_proj["cell_y"] = (gdf_medidores_proj["y"] // CELL_SIZE_M).astype(int)
gdf_medidores_proj["cell_id"] = (
    gdf_medidores_proj["cell_x"].astype(str)
    + "_"
    + gdf_medidores_proj["cell_y"].astype(str)
)

coords_global = gdf_medidores_proj[["x", "y"]].values
tree_global = BallTree(coords_global, metric="euclidean")

dist_global, _ = tree_global.query(coords_global, k=6)
umbral_global_m = np.percentile(dist_global[:, 5], 90)

print("Umbral global de respaldo:", round(umbral_global_m, 2), "m")

umbrales_celda = {}

for cell_id, grupo in gdf_medidores_proj.groupby("cell_id"):
    if len(grupo) >= 6:
        coords_celda = grupo[["x", "y"]].values
        tree_celda = BallTree(coords_celda, metric="euclidean")
        dist_celda, _ = tree_celda.query(coords_celda, k=6)

        umbral_celda = np.percentile(dist_celda[:, 5], 90)
    else:
        umbral_celda = umbral_global_m

    umbrales_celda[cell_id] = umbral_celda

gdf_medidores_proj["radio_candidatos_zona_m"] = (
    gdf_medidores_proj["cell_id"].map(umbrales_celda)
)

medidores["cell_id"] = gdf_medidores_proj["cell_id"].values
medidores["radio_candidatos_zona_m"] = gdf_medidores_proj["radio_candidatos_zona_m"].values

print("Resumen de radios adaptativos:")
medidores["radio_candidatos_zona_m"].describe()

Umbral global de respaldo: 299.77 m
Resumen de radios adaptativos:


,radio_candidatos_zona_m
count,5072.000000
mean,338.712879
std,125.098680
min,175.879512
25%,260.652219
50%,303.376917
75%,374.644799
max,1134.674510


## Análisis de distancias entre medidores y generación de pares candidatos

In [23]:
# Coordenadas en radianes para distancia haversine
coords = np.radians(medidores[["latitud", "longitud"]].values)

tree = BallTree(coords, metric="haversine")

# Calculamos hasta los 20 vecinos más cercanos para estudiar la distribución
K_ANALISIS = 20
distancias, indices = tree.query(coords, k=K_ANALISIS + 1)

R = 6371000  # radio tierra metros
distancias_m = distancias * R

In [24]:
resumen_vecinos = pd.DataFrame({
    "vecino_1_m": distancias_m[:, 1],
    "vecino_3_m": distancias_m[:, 3],
    "vecino_5_m": distancias_m[:, 5],
    "vecino_10_m": distancias_m[:, 10],
    "vecino_20_m": distancias_m[:, 20],
})

resumen_vecinos.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95])

,vecino_1_m,vecino_3_m,vecino_5_m,vecino_10_m,vecino_20_m
count,5072.000000,5072.000000,5072.000000,5072.000000,5072.000000
mean,60.299512,134.802462,192.218004,303.847693,479.846961
std,50.277204,83.244571,106.243956,150.109013,302.088483
min,0.000000,10.896282,15.656927,80.867585,195.699433
25%,16.317860,89.649261,130.094414,216.112762,344.834643
50%,51.730719,125.062705,172.480334,274.661836,421.074841
75%,90.709754,164.091985,225.365385,348.903363,527.182705
90%,123.780820,211.121938,299.572190,449.004668,660.410756
95%,148.631821,252.537244,367.262305,544.203109,817.526027
max,559.979456,1626.513574,1656.200417,1741.671511,4377.116647


In [25]:
radio_candidatos_m = np.percentile(distancias_m[:, 5], 90)

print("Radio de candidatos calculado:", radio_candidatos_m, "metros")

Radio de candidatos calculado: 299.57218999318764 metros


In [26]:
# Crear pares candidatos usando el umbral adaptativo por zona

coords_proj = gdf_medidores_proj[["x", "y"]].values
tree = BallTree(coords_proj, metric="euclidean")

radio_max_m = medidores["radio_candidatos_zona_m"].max()

indices_radio, distancias_radio = tree.query_radius(
    coords_proj,
    r=radio_max_m,
    return_distance=True,
    sort_results=True
)

pares_candidatos = []

for i in range(len(medidores)):
    medidor_origen = medidores.iloc[i]
    radio_origen = medidor_origen["radio_candidatos_zona_m"]

    for j, distancia_m in zip(indices_radio[i], distancias_radio[i]):
        if i == j:
            continue

        medidor_destino = medidores.iloc[j]
        radio_destino = medidor_destino["radio_candidatos_zona_m"]

        radio_par_m = max(radio_origen, radio_destino)

        if distancia_m > radio_par_m:
            continue

        pares_candidatos.append({
            "id_origen": medidor_origen["id"],
            "id_destino": medidor_destino["id"],
            "distancia_directa_m": distancia_m,
            "osm_node_origen": medidor_origen["osm_node"],
            "osm_node_destino": medidor_destino["osm_node"],
            "cell_id_origen": medidor_origen["cell_id"],
            "cell_id_destino": medidor_destino["cell_id"],
            "radio_origen_m": radio_origen,
            "radio_destino_m": radio_destino
        })

df_pares = pd.DataFrame(pares_candidatos).drop_duplicates()

print("Pares candidatos con umbral adaptativo:", len(df_pares))
df_pares.head()

Pares candidatos con umbral adaptativo: 78514


,id_origen,id_destino,distancia_directa_m,osm_node_origen,osm_node_destino,cell_id_origen,cell_id_destino,radio_origen_m,radio_destino_m
0,6835,6833,20.144904,32636471,32636471,438_4474,438_4474,282.529019,282.529019
1,6835,6836,91.777636,32636471,315261895,438_4474,438_4474,282.529019,282.529019
2,6835,6837,94.127890,32636471,315264896,438_4474,438_4474,282.529019,282.529019
3,6835,6827,110.782583,32636471,315261895,438_4474,438_4474,282.529019,282.529019
4,6835,1042,116.942125,32636471,315265031,438_4474,438_4474,282.529019,282.529019


In [27]:
candidatos_por_medidor = (
    df_pares
    .groupby("id_origen")
    .size()
    .reset_index(name="num_candidatos")
)

candidatos_por_medidor["num_candidatos"].describe()


,num_candidatos
count,5069.000000
mean,15.489051
std,8.260420
min,1.000000
25%,10.000000
50%,14.000000
75%,19.000000
max,85.000000


## Cálculo de caminos reales sobre la red OSM

In [28]:
from tqdm import tqdm
aristas_reales = []

for _, row in tqdm(df_pares.iterrows(), total=len(df_pares)):
    id_origen = int(row["id_origen"])
    id_destino = int(row["id_destino"])

    osm_origen = int(row["osm_node_origen"])
    osm_destino = int(row["osm_node_destino"])

    distancia_directa_m = float(row["distancia_directa_m"])

    # Caso especial: dos medidores asociados al mismo nodo OSM
    if osm_origen == osm_destino:
        aristas_reales.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": 0.0,
            "tipo_conexion": "mismo_nodo_osm"
        })
        continue

    try:
        distancia_red_m = nx.shortest_path_length(
            G_osm,
            source=osm_origen,
            target=osm_destino,
            weight="length"
        )

        aristas_reales.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": float(distancia_red_m),
            "tipo_conexion": "camino_osm"
        })

    except (nx.NetworkXNoPath, nx.NodeNotFound):
        # Si no hay camino real en OSM, no se crea arista
        continue

df_aristas_reales = pd.DataFrame(aristas_reales).drop_duplicates()

print("Pares candidatos:", len(df_pares))
print("Aristas reales encontradas:", len(df_aristas_reales))

df_aristas_reales.head()

100%|██████████| 78514/78514 [01:38<00:00, 799.85it/s] 


Pares candidatos: 78514
Aristas reales encontradas: 78212


,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion
0,6835,6833,32636471,32636471,20.144904,0.000000,mismo_nodo_osm
1,6835,6836,32636471,315261895,91.777636,6655.017406,camino_osm
2,6835,6837,32636471,315264896,94.127890,2248.920678,camino_osm
3,6835,6827,32636471,315261895,110.782583,6655.017406,camino_osm
4,6835,1042,32636471,315265031,116.942125,2583.076553,camino_osm


## Validación de aristas
Se calcula el factor de rodeo:

\[
factor\_rodeo = \frac{distancia\_red\_m}{distancia\_directa\_m}
\]

Este factor permite detectar conexiones donde dos medidores están cerca en línea recta, pero el camino real por carretera es mucho más largo.  
Estas aristas no se eliminan automáticamente: se marcan para revisión.

In [29]:
df_aristas_reales["factor_rodeo"] = np.where(
    df_aristas_reales["distancia_directa_m"] > 0,
    df_aristas_reales["distancia_red_m"] / df_aristas_reales["distancia_directa_m"],
    np.nan
)

# Si están en el mismo nodo OSM, consideramos factor de rodeo 0
df_aristas_reales.loc[
    df_aristas_reales["tipo_conexion"] == "mismo_nodo_osm",
    "factor_rodeo"
] = 0

df_aristas_reales[[
    "distancia_directa_m",
    "distancia_red_m",
    "factor_rodeo"
]].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])

,distancia_directa_m,distancia_red_m,factor_rodeo
count,78212.000000,78212.000000,78212.000000
mean,261.265802,752.804330,3.731141
std,171.738788,1101.373915,11.028646
min,0.000000,0.000000,0.000000
50%,227.564055,441.484740,1.601172
75%,333.214575,782.677871,2.899921
90%,476.344446,1561.313707,6.555969
95%,612.303740,2790.882019,11.906756
99%,849.055759,5283.058836,38.886467
max,1133.737840,13963.539403,736.169229


In [30]:
# Umbrales
umbral_distancia_red = df_aristas_reales["distancia_red_m"].quantile(0.95)
umbral_rodeo = df_aristas_reales["factor_rodeo"].quantile(0.95)

print("Umbral distancia red P95:", umbral_distancia_red)
print("Umbral rodeo P95:", umbral_rodeo)

Umbral distancia red P95: 2790.88201916928
Umbral rodeo P95: 11.906756054919533


In [31]:
df_aristas_reales["revisar_distancia_red_alta"] = (
    df_aristas_reales["distancia_red_m"] > umbral_distancia_red
)

df_aristas_reales["revisar_rodeo_alto"] = (
    df_aristas_reales["factor_rodeo"] > umbral_rodeo
)

def clasificar_arista(row):
    if row["revisar_distancia_red_alta"] and row["revisar_rodeo_alto"]:
        return "revisar_distancia_y_rodeo"
    elif row["revisar_rodeo_alto"]:
        return "revisar_rodeo_alto"
    elif row["revisar_distancia_red_alta"]:
        return "revisar_distancia_red_alta"
    else:
        return "ok"

df_aristas_reales["calidad_arista"] = df_aristas_reales.apply(clasificar_arista, axis=1)

df_aristas_reales["calidad_arista"].value_counts()

,count
calidad_arista,
ok,72897
revisar_distancia_y_rodeo,2504
revisar_rodeo_alto,1407
revisar_distancia_red_alta,1404


In [32]:
# Aristas con mayor factor de rodeo
df_aristas_reales.sort_values("factor_rodeo", ascending=False).head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,revisar_distancia_red_alta,revisar_rodeo_alto,calidad_arista
74662,6952,6949,5360984763,307997125,12.229916,9003.287558,camino_osm,736.169229,True,True,revisar_distancia_y_rodeo
60220,1015,1016,338920029,21723233,8.800137,6123.576228,camino_osm,695.850129,True,True,revisar_distancia_y_rodeo
12084,11370,6786,25549913,297767512,13.579016,6413.325810,camino_osm,472.296795,True,True,revisar_distancia_y_rodeo
74663,6952,6950,5360984763,307997125,19.531362,9003.287558,camino_osm,460.965687,True,True,revisar_distancia_y_rodeo
12118,1052,1049,388087148,2493682299,25.667519,11766.287184,camino_osm,458.411558,True,True,revisar_distancia_y_rodeo
74665,6951,6949,5360984763,307997125,19.884963,9003.287558,camino_osm,452.768641,True,True,revisar_distancia_y_rodeo
1349,11373,11374,9823064711,429461526,18.509183,6765.444483,camino_osm,365.518258,True,True,revisar_distancia_y_rodeo
73665,3826,6790,21525883,20953256,6.949251,2341.325133,camino_osm,336.917625,False,True,revisar_rodeo_alto
1350,11373,11375,9823064711,429461526,21.163492,6765.444483,camino_osm,319.675246,True,True,revisar_distancia_y_rodeo
33900,6738,6737,25938853,2537144683,18.550397,5833.573622,camino_osm,314.471625,True,True,revisar_distancia_y_rodeo


In [33]:
# Aristas con mayor distancia real sobre red
df_aristas_reales.sort_values("distancia_red_m", ascending=False).head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,revisar_distancia_red_alta,revisar_rodeo_alto,calidad_arista
64564,1035,6858,315244935,2136384930,206.703195,13963.539403,camino_osm,67.553573,True,True,revisar_distancia_y_rodeo
64573,1035,1032,315244935,2136384930,227.777276,13963.539403,camino_osm,61.303479,True,True,revisar_distancia_y_rodeo
64572,1035,7145,315244935,2136384930,217.058327,13963.539403,camino_osm,64.330817,True,True,revisar_distancia_y_rodeo
64563,1035,6857,315244935,2136384930,204.603830,13963.539403,camino_osm,68.246716,True,True,revisar_distancia_y_rodeo
64575,1035,7118,315244935,2136384930,237.664874,13963.539403,camino_osm,58.753063,True,True,revisar_distancia_y_rodeo
64568,1035,6856,315244935,2136384930,215.235601,13963.539403,camino_osm,64.875603,True,True,revisar_distancia_y_rodeo
12933,11496,6857,315244935,2136384930,208.449331,13963.539403,camino_osm,66.987691,True,True,revisar_distancia_y_rodeo
12946,11496,7118,315244935,2136384930,240.349550,13963.539403,camino_osm,58.096799,True,True,revisar_distancia_y_rodeo
12942,11496,6856,315244935,2136384930,219.088382,13963.539403,camino_osm,63.734732,True,True,revisar_distancia_y_rodeo
12936,11496,6858,315244935,2136384930,210.144953,13963.539403,camino_osm,66.447179,True,True,revisar_distancia_y_rodeo


In [34]:
df_aristas_reales

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,revisar_distancia_red_alta,revisar_rodeo_alto,calidad_arista
0,6835,6833,32636471,32636471,20.144904,0.000000,mismo_nodo_osm,0.000000,False,False,ok
1,6835,6836,32636471,315261895,91.777636,6655.017406,camino_osm,72.512408,True,True,revisar_distancia_y_rodeo
2,6835,6837,32636471,315264896,94.127890,2248.920678,camino_osm,23.892182,False,True,revisar_rodeo_alto
3,6835,6827,32636471,315261895,110.782583,6655.017406,camino_osm,60.072777,True,True,revisar_distancia_y_rodeo
4,6835,1042,32636471,315265031,116.942125,2583.076553,camino_osm,22.088504,False,True,revisar_rodeo_alto
...,...,...,...,...,...,...,...,...,...,...,...
78207,10469,5472,1754817102,25902910,336.537641,501.017560,camino_osm,1.488742,False,False,ok
78208,10469,4459,1754817102,98949534,362.323556,1072.431946,camino_osm,2.959874,False,False,ok
78209,10469,4469,1754817102,25903360,364.270639,1316.301680,camino_osm,3.613527,False,False,ok
78210,10469,9965,1754817102,989204859,370.848768,1016.889895,camino_osm,2.742061,False,False,ok


## Construcción grafo

In [35]:
G_medidores = nx.DiGraph()

# Añadir todos los medidores reales como nodos
for _, row in medidores.iterrows():
    G_medidores.add_node(
        int(row["id"]),
        nombre=row["nombre"] if pd.notna(row["nombre"]) else "sin_nombre",
        latitud=float(row["latitud"]),
        longitud=float(row["longitud"]),
        osm_node=int(row["osm_node"]),
        distancia_osm_node_m=float(row["distancia_osm_node_m"])
    )

# Añadir aristas reales calculadas sobre OSM
for _, row in df_aristas_reales.iterrows():
    G_medidores.add_edge(
        int(row["id_origen"]),
        int(row["id_destino"]),
        distancia_directa_m=float(row["distancia_directa_m"]),
        distancia_red_m=float(row["distancia_red_m"]),
        factor_rodeo=float(row["factor_rodeo"]),
        weight=float(row["distancia_red_m"]),
        tipo_conexion=row["tipo_conexion"],
        calidad_arista=row["calidad_arista"]
    )

print("Nodos:", G_medidores.number_of_nodes())
print("Aristas:", G_medidores.number_of_edges())

Nodos: 5072
Aristas: 78212


In [36]:
nodos_con_aristas = set(df_aristas_reales["id_origen"]).union(
    set(df_aristas_reales["id_destino"])
)

nodos_aislados = set(medidores["id"]) - nodos_con_aristas

print("Medidores totales:", medidores["id"].nunique())
print("Medidores con alguna arista:", len(nodos_con_aristas))
print("Medidores aislados:", len(nodos_aislados))

print("Componentes débiles:", nx.number_weakly_connected_components(G_medidores))
print("Componentes fuertes:", nx.number_strongly_connected_components(G_medidores))

Medidores totales: 5072
Medidores con alguna arista: 5069
Medidores aislados: 3
Componentes débiles: 25
Componentes fuertes: 43


In [37]:
medidores_aislados = medidores[
    medidores["id"].isin(nodos_aislados)
].copy()

medidores_aislados[
    ["id", "nombre", "latitud", "longitud", "osm_node", "distancia_osm_node_m"]
].sort_values("distancia_osm_node_m", ascending=False)

,id,nombre,latitud,longitud,osm_node,distancia_osm_node_m
2302,6489,Embajadores - Santa Catalina-Carretera Villave...,40.369021,-3.676004,306101165,90.843345
3214,4868,(TACTICO) BATALLA GARELLANO Nº 27 S-N (SIRRACH...,40.455164,-3.793131,292702537,15.418926
4482,6490,Embajadores - Av. Santa Catalina-Caja Mágica,40.369531,-3.679801,2314755080,11.016507


## VISUALIZACIÓN DE NODOS EN EL MAPA

In [38]:
import folium
from folium.plugins import MarkerCluster

# Centro del mapa en la media de coordenadas de los medidores
lat_centro = medidores["latitud"].mean()
lon_centro = medidores["longitud"].mean()

mapa = folium.Map(
    location=[lat_centro, lon_centro],
    zoom_start=13,
    tiles="CartoDB positron"   # fondo claro, las aristas se ven mejor
)

# Verde  → medidor con aristas OK
# Rojo   → medidor aislado (sin aristas)

nodos_con_aristas_viz = set(df_aristas_reales["id_origen"]).union(
    set(df_aristas_reales["id_destino"])
)

cluster = MarkerCluster(name="Puntos medidores", show=True)

for _, row in medidores.iterrows():
    es_aislado = int(row["id"]) not in nodos_con_aristas_viz
    color = "red" if es_aislado else "green"
    icono  = "times" if es_aislado else "circle"

    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>ID:</b> {int(row['id'])}<br>"
            f"<b>Nombre:</b> {row['nombre']}<br>"
            f"<b>OSM node:</b> {int(row['osm_node'])}<br>"
            f"<b>Dist. a OSM:</b> {row['distancia_osm_node_m']:.1f} m<br>"
            f"<b>Estado:</b> {'🔴 Aislado' if es_aislado else '🟢 Conectado'}",
            max_width=250
        ),
        tooltip=f"ID {int(row['id'])} – {row['nombre']}"
    ).add_to(cluster)

cluster.add_to(mapa)

In [39]:
coord_por_id = medidores.set_index("id")[["latitud", "longitud"]].to_dict("index")

# Paleta de colores por calidad de arista
COLOR_ARISTA = {
    "ok":                         "#2ecc71",   # verde
    "revisar_rodeo_alto":         "#f39c12",   # naranja
    "revisar_distancia_red_alta": "#3498db",   # azul
    "revisar_distancia_y_rodeo":  "#e74c3c",   # rojo
    "mismo_nodo_osm":             "#9b59b6",   # morado
}

capa_ok      = folium.FeatureGroup(name="Aristas OK",            show=True)
capa_rodeo   = folium.FeatureGroup(name="Revisar rodeo",         show=True)
capa_dist    = folium.FeatureGroup(name="Revisar distancia",     show=True)
capa_ambos   = folium.FeatureGroup(name="Revisar dist+rodeo",   show=False)
capa_mismo   = folium.FeatureGroup(name="Mismo nodo OSM",        show=False)

CAPA_MAP = {
    "ok":                         capa_ok,
    "revisar_rodeo_alto":         capa_rodeo,
    "revisar_distancia_red_alta": capa_dist,
    "revisar_distancia_y_rodeo":  capa_ambos,
    "mismo_nodo_osm":             capa_mismo,
}

for _, row in df_aristas_reales.iterrows():
    id_o = int(row["id_origen"])
    id_d = int(row["id_destino"])

    if id_o not in coord_por_id or id_d not in coord_por_id:
        continue

    p_origen  = (coord_por_id[id_o]["latitud"], coord_por_id[id_o]["longitud"])
    p_destino = (coord_por_id[id_d]["latitud"], coord_por_id[id_d]["longitud"])

    calidad = row["calidad_arista"]
    color   = COLOR_ARISTA.get(calidad, "#95a5a6")
    capa    = CAPA_MAP.get(calidad, capa_ok)

    folium.PolyLine(
        locations=[p_origen, p_destino],
        color=color,
        weight=1.5,
        opacity=0.6,
        tooltip=(
            f"ID {id_o} → {id_d} | "
            f"Red: {row['distancia_red_m']:.0f} m | "
            f"Rodeo: {row['factor_rodeo']:.2f}x | "
            f"{calidad}"
        )
    ).add_to(capa)

for capa in [capa_ok, capa_rodeo, capa_dist, capa_ambos, capa_mismo]:
    capa.add_to(mapa)

In [40]:
# Análisis de componentes débiles del grafo de medidores

componentes = list(nx.weakly_connected_components(G_medidores))
componentes_ordenados = sorted(componentes, key=len, reverse=True)

print("Total de componentes débiles:", len(componentes_ordenados))
print()
print("Tamaño de cada componente:")

for i, comp in enumerate(componentes_ordenados):
    print(f"Componente {i + 1}: {len(comp)} nodos")

Total de componentes débiles: 25

Tamaño de cada componente:
Componente 1: 4790 nodos
Componente 2: 64 nodos
Componente 3: 43 nodos
Componente 4: 39 nodos
Componente 5: 25 nodos
Componente 6: 17 nodos
Componente 7: 13 nodos
Componente 8: 12 nodos
Componente 9: 11 nodos
Componente 10: 8 nodos
Componente 11: 7 nodos
Componente 12: 6 nodos
Componente 13: 5 nodos
Componente 14: 5 nodos
Componente 15: 4 nodos
Componente 16: 4 nodos
Componente 17: 3 nodos
Componente 18: 3 nodos
Componente 19: 3 nodos
Componente 20: 3 nodos
Componente 21: 2 nodos
Componente 22: 2 nodos
Componente 23: 1 nodos
Componente 24: 1 nodos
Componente 25: 1 nodos


In [41]:
# Ver qué zonas son los componentes más grandes fuera del componente principal

comp_principal = componentes_ordenados[0]

print("Componentes desconectados más grandes fuera del principal:")
print()

for i, comp in enumerate(componentes_ordenados[1:11], start=2):
    nodos_comp = list(comp)

    lats = [G_medidores.nodes[n]["latitud"] for n in nodos_comp]
    lons = [G_medidores.nodes[n]["longitud"] for n in nodos_comp]
    nombres = [G_medidores.nodes[n]["nombre"] for n in nodos_comp[:3]]

    print(f"Componente {i} ({len(comp)} nodos)")
    print(f"Centro aproximado: lat {np.mean(lats):.4f}, lon {np.mean(lons):.4f}")
    print(f"Ejemplos: {nombres}")
    print()

Componentes desconectados más grandes fuera del principal:

Componente 2 (64 nodos)
Centro aproximado: lat 40.4571, lon -3.7827
Ejemplos: ['(MICRO) ANA TERESA, 29A O-E (A. VINDEL - PLEYADES)', 'PLEYADES Nº 19 S-N (ANA TERESA - ARDALES)', '(TACTICO) ARANDIGA Nº 8 N-S (ARASCUES - AV. OSA MAYOR)']

Componente 3 (43 nodos)
Centro aproximado: lat 40.3672, lon -3.6045
Ejemplos: ['José Gutiérrez - Av. Suertes-Av. Ensanche de Vallecas', 'Av. Ensanche de Vallecas - Rafael Leon-Cañada Santisimo', 'Cañada Santisimo - Av. Cerro Milano-Av. Ensanche de Vallecas']

Componente 4 (39 nodos)
Centro aproximado: lat 40.4694, lon -3.7431
Ejemplos: ['ARROYOFRESNO S-N (CANTALEJO-AV. HERRERA ORIA)', 'AV. CARDENAL HERRERA ORIA O-E (VEGAFRIA - ARROYOFRESNO)', 'PM22571']

Componente 5 (25 nodos)
Centro aproximado: lat 40.4672, lon -3.5862
Ejemplos: ['Av. Logroño, 160 - Alhaurin-Playa de Riazor', 'Av. Logroño 347 - Playa de Riazor-Alhaurin', 'TÁCTICO - Alhaurín 19 - Arroyomolino-Av. Logroño']

Componente 6 (17 no

In [42]:
# Distancia mínima entre los componentes desconectados y el componente principal

print("Distancia mínima entre componentes desconectados y el componente principal:")
print()

nodos_principal = list(comp_principal)

coords_principal = np.radians([
    [
        G_medidores.nodes[n]["latitud"],
        G_medidores.nodes[n]["longitud"]
    ]
    for n in nodos_principal
])

tree_principal = BallTree(coords_principal, metric="haversine")

for i, comp in enumerate(componentes_ordenados[1:11], start=2):
    nodos_comp = list(comp)

    coords_comp = np.radians([
        [
            G_medidores.nodes[n]["latitud"],
            G_medidores.nodes[n]["longitud"]
        ]
        for n in nodos_comp
    ])

    dists, _ = tree_principal.query(coords_comp, k=1)
    dist_min = dists.min() * 6371000

    print(
        f"Componente {i} ({len(comp)} nodos): "
        f"distancia mínima al principal = {dist_min:.0f} m"
    )

Distancia mínima entre componentes desconectados y el componente principal:

Componente 2 (64 nodos): distancia mínima al principal = 3093 m
Componente 3 (43 nodos): distancia mínima al principal = 1174 m
Componente 4 (39 nodos): distancia mínima al principal = 489 m
Componente 5 (25 nodos): distancia mínima al principal = 1459 m
Componente 6 (17 nodos): distancia mínima al principal = 2523 m
Componente 7 (13 nodos): distancia mínima al principal = 1356 m
Componente 8 (12 nodos): distancia mínima al principal = 425 m
Componente 9 (11 nodos): distancia mínima al principal = 459 m
Componente 10 (8 nodos): distancia mínima al principal = 8291 m
Componente 11 (7 nodos): distancia mínima al principal = 615 m


In [43]:
# Guardar resumen de componentes

resumen_componentes = []

for i, comp in enumerate(componentes_ordenados):
    nodos_comp = list(comp)

    lats = [G_medidores.nodes[n]["latitud"] for n in nodos_comp]
    lons = [G_medidores.nodes[n]["longitud"] for n in nodos_comp]

    if i == 0:
        dist_min_principal = 0
    else:
        coords_comp = np.radians([
            [
                G_medidores.nodes[n]["latitud"],
                G_medidores.nodes[n]["longitud"]
            ]
            for n in nodos_comp
        ])

        dists, _ = tree_principal.query(coords_comp, k=1)
        dist_min_principal = round(dists.min() * 6371000, 1)

    resumen_componentes.append({
        "componente": i + 1,
        "num_nodos": len(comp),
        "lat_centro": round(np.mean(lats), 4),
        "lon_centro": round(np.mean(lons), 4),
        "dist_min_al_principal_m": dist_min_principal,
        "es_principal": i == 0
    })

df_componentes = pd.DataFrame(resumen_componentes)

df_componentes.to_csv(
    PROCESSED_DIR / "analisis_componentes.csv",
    index=False,
    encoding="utf-8"
)

print("Guardado en:", PROCESSED_DIR / "analisis_componentes.csv")
df_componentes.head(15)

Guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/processed/analisis_componentes.csv


,componente,num_nodos,lat_centro,lon_centro,dist_min_al_principal_m,es_principal
0,1,4790,40.4302,-3.6838,0.0,True
1,2,64,40.4571,-3.7827,3093.1,False
2,3,43,40.3672,-3.6045,1173.7,False
3,4,39,40.4694,-3.7431,489.2,False
4,5,25,40.4672,-3.5862,1459.5,False
5,6,17,40.4064,-3.5569,2523.0,False
6,7,13,40.4442,-3.5846,1356.0,False
7,8,12,40.4480,-3.7275,424.8,False
8,9,11,40.4767,-3.6383,458.9,False
9,10,8,40.4737,-3.8332,8291.0,False


In [44]:
# Mapa de componentes

mapa_componentes = folium.Map(
    location=[
        medidores["latitud"].mean(),
        medidores["longitud"].mean()
    ],
    zoom_start=11
)

# Colores para los primeros componentes
colores = [
    "blue", "red", "green", "purple", "orange",
    "darkred", "lightred", "beige", "darkblue", "darkgreen",
    "cadetblue", "pink", "lightblue", "lightgreen", "gray"
]

# Pintamos solo los primeros 15 componentes para que el mapa sea legible
for i, comp in enumerate(componentes_ordenados[:15]):
    color = colores[i % len(colores)]
    etiqueta = "Principal" if i == 0 else f"Comp {i + 1} ({len(comp)} nodos)"

    for nodo in comp:
        lat = G_medidores.nodes[nodo]["latitud"]
        lon = G_medidores.nodes[nodo]["longitud"]
        nombre = G_medidores.nodes[nodo]["nombre"]

        folium.CircleMarker(
            location=[lat, lon],
            radius=3 if i == 0 else 5,
            color=color,
            fill=True,
            fill_opacity=0.6 if i == 0 else 0.9,
            popup=f"[{etiqueta}]<br>ID: {nodo}<br>{nombre}"
        ).add_to(mapa_componentes)

ruta_mapa_componentes = PROCESSED_DIR / "mapa_componentes.html"
mapa_componentes.save(ruta_mapa_componentes)

print("Mapa guardado en:", ruta_mapa_componentes.resolve())

Mapa guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/processed/mapa_componentes.html


In [45]:
from IPython.display import display

display(mapa_componentes)

In [46]:
print("Nodos:", G_medidores.number_of_nodes())
print("Aristas:", G_medidores.number_of_edges())
print("Componentes débiles:", nx.number_weakly_connected_components(G_medidores))
print("Componentes fuertes:", nx.number_strongly_connected_components(G_medidores))

Nodos: 5072
Aristas: 78212
Componentes débiles: 25
Componentes fuertes: 43


## Integración de velocidades en el grafo



In [47]:
from pathlib import Path
import geopandas as gpd

BASE_DIR = Path("/content/drive/MyDrive/Movilidad_inteligente_madrid")
VELOCIDADES_DIR = BASE_DIR / "mapa_velocidades"

ruta_velocidades = VELOCIDADES_DIR / "velocidades_madrid_final.geojson"

print("Ruta velocidades:", ruta_velocidades)
print("Existe:", ruta_velocidades.exists())

gdf_velocidades_full = gpd.read_file(ruta_velocidades)

print("Registros:", len(gdf_velocidades_full))
print("Columnas disponibles:")
print(gdf_velocidades_full.columns.tolist())

gdf_velocidades_full.head()

Ruta velocidades: /content/drive/MyDrive/Movilidad_inteligente_madrid/mapa_velocidades/velocidades_madrid_final.geojson
Existe: True
Registros: 271232
Columnas disponibles:
['u', 'v', 'key', 'highway', 'maxspeed', 'es_urbano', 'maxspeed_final', 'geometry']


,u,v,key,highway,maxspeed,es_urbano,maxspeed_final,geometry
0,21741584,21741587,0,[motorway],[100],False,[100],"LINESTRING (-3.69059 40.26517, -3.6905 40.2628..."
1,21741587,759985345,0,[motorway_link],"[40, 70]",False,"[40, 70]","LINESTRING (-3.69004 40.24808, -3.6901 40.2477..."
2,21741587,310025091,0,[motorway],[100],False,[100],"LINESTRING (-3.69004 40.24808, -3.68994 40.245..."
3,21741595,1317348460,0,"[motorway_link, tertiary]",[40],True,[40],"LINESTRING (-3.67595 40.1961, -3.6759 40.19589..."
4,21741595,310031145,0,[motorway],"[100, 120]",False,"[100, 120]","LINESTRING (-3.67595 40.1961, -3.67566 40.1956..."


In [48]:
gdf_velocidades = gdf_velocidades_full.copy()

columnas_necesarias = ["u", "v", "maxspeed_final"]

for col in columnas_necesarias:
    if col not in gdf_velocidades.columns:
        raise ValueError(f"Falta la columna necesaria: {col}")

print("Columnas conservadas:", len(gdf_velocidades.columns))
gdf_velocidades[["u", "v", "maxspeed_final"]].head()

Columnas conservadas: 8


,u,v,maxspeed_final
0,21741584,21741587,[100]
1,21741587,759985345,"[40, 70]"
2,21741587,310025091,[100]
3,21741595,1317348460,[40]
4,21741595,310031145,"[100, 120]"


In [49]:
import ast
import re

def extraer_numeros_velocidad(valor):
    if valor is None:
        return []

    if isinstance(valor, float) and np.isnan(valor):
        return []

    if isinstance(valor, (int, float, np.integer, np.floating)):
        return [float(valor)]

    if isinstance(valor, (list, tuple, set, np.ndarray)):
        numeros = []
        for item in valor:
            numeros.extend(extraer_numeros_velocidad(item))
        return numeros

    texto = str(valor).strip()

    if texto.lower() in ["", "nan", "none"]:
        return []

    try:
        valor_parseado = ast.literal_eval(texto)
        return extraer_numeros_velocidad(valor_parseado)
    except Exception:
        pass

    numeros = re.findall(r"\d+(?:[.,]\d+)?", texto)
    return [float(n.replace(",", ".")) for n in numeros]


def agregar_velocidad(valor, metodo="media"):
    numeros = extraer_numeros_velocidad(valor)
    numeros = [n for n in numeros if 0 < n <= 150]

    if len(numeros) == 0:
        return np.nan

    if metodo == "media":
        return float(np.mean(numeros))
    elif metodo == "mediana":
        return float(np.median(numeros))
    elif metodo == "max":
        return float(np.max(numeros))
    elif metodo == "min":
        return float(np.min(numeros))
    else:
        raise ValueError("Método no válido")


METODO_AGREGACION_VELOCIDAD = "media"

gdf_velocidades["velocidad_kmh"] = gdf_velocidades["maxspeed_final"].apply(
    lambda x: agregar_velocidad(x, metodo=METODO_AGREGACION_VELOCIDAD)
)

print("Filas totales:", len(gdf_velocidades))
print("Velocidades válidas:", gdf_velocidades["velocidad_kmh"].notna().sum())

gdf_velocidades.head()

Filas totales: 271232
Velocidades válidas: 271209


,u,v,key,highway,maxspeed,es_urbano,maxspeed_final,geometry,velocidad_kmh
0,21741584,21741587,0,[motorway],[100],False,[100],"LINESTRING (-3.69059 40.26517, -3.6905 40.2628...",100.0
1,21741587,759985345,0,[motorway_link],"[40, 70]",False,"[40, 70]","LINESTRING (-3.69004 40.24808, -3.6901 40.2477...",55.0
2,21741587,310025091,0,[motorway],[100],False,[100],"LINESTRING (-3.69004 40.24808, -3.68994 40.245...",100.0
3,21741595,1317348460,0,"[motorway_link, tertiary]",[40],True,[40],"LINESTRING (-3.67595 40.1961, -3.6759 40.19589...",40.0
4,21741595,310031145,0,[motorway],"[100, 120]",False,"[100, 120]","LINESTRING (-3.67595 40.1961, -3.67566 40.1956...",110.0


In [50]:
# Preparar u y v
gdf_velocidades["u"] = pd.to_numeric(gdf_velocidades["u"], errors="coerce")
gdf_velocidades["v"] = pd.to_numeric(gdf_velocidades["v"], errors="coerce")

gdf_velocidades_validas = gdf_velocidades.dropna(subset=["u", "v"]).copy()

gdf_velocidades_validas["u"] = gdf_velocidades_validas["u"].astype(int)
gdf_velocidades_validas["v"] = gdf_velocidades_validas["v"].astype(int)

columnas_atributos = [
    col for col in gdf_velocidades_validas.columns
    if col not in ["u", "v", "geometry"]
]

print("Columnas de atributos que se van a conservar:")
print(columnas_atributos)

# Diccionario (u, v) todos los atributos del tramo
atributos_por_arista = (
    gdf_velocidades_validas
    .drop_duplicates(subset=["u", "v"])
    .set_index(["u", "v"])[columnas_atributos]
    .to_dict(orient="index")
)

print("Tramos con atributos disponibles:", len(atributos_por_arista))

Columnas de atributos que se van a conservar:
['key', 'highway', 'maxspeed', 'es_urbano', 'maxspeed_final', 'velocidad_kmh']
Tramos con atributos disponibles: 269506


In [51]:
# Añadir atributos del GeoJSON completo a las aristas de G_osm

aristas_con_atributos = 0
aristas_sin_atributos = 0

for u, v, k, data in G_osm.edges(keys=True, data=True):
    u_int = int(u)
    v_int = int(v)

    atributos = atributos_por_arista.get((u_int, v_int))

    if atributos is None:
        aristas_sin_atributos += 1
        data["velocidad_disponible"] = False
        data["fuente_velocidad"] = "sin_atributos_geojson"
        data["velocidad_kmh"] = np.nan
        data["travel_time_s"] = np.nan
        continue

    aristas_con_atributos += 1


    for nombre_columna, valor in atributos.items():
        if isinstance(valor, (list, tuple, set, dict)):
            valor = str(valor)

        data[f"vel_{nombre_columna}"] = valor

    #velocidad_kmh solo si existe realmente
    velocidad = atributos.get("velocidad_kmh")

    if velocidad is not None and pd.notna(velocidad):
        velocidad = float(velocidad)
        longitud_m = float(data.get("length", 0))
        tiempo_s = longitud_m / (velocidad * 1000 / 3600) if longitud_m > 0 else np.nan

        data["velocidad_kmh"] = velocidad
        data["travel_time_s"] = tiempo_s
        data["fuente_velocidad"] = "geojson_inferencia"
        data["velocidad_disponible"] = True
    else:
        data["velocidad_kmh"] = np.nan
        data["travel_time_s"] = np.nan
        data["fuente_velocidad"] = "sin_velocidad_inferida"
        data["velocidad_disponible"] = False

print("Aristas OSM con atributos del GeoJSON:", aristas_con_atributos)
print("Aristas OSM sin atributos del GeoJSON:", aristas_sin_atributos)

Aristas OSM con atributos del GeoJSON: 61804
Aristas OSM sin atributos del GeoJSON: 39


In [52]:
print("G_osm existe:", "G_osm" in globals())
print("df_pares existe:", "df_pares" in globals())
print("medidores existe:", "medidores" in globals())

print("Aristas G_osm:", G_osm.number_of_edges())
print("Pares candidatos:", len(df_pares))
print("Medidores:", len(medidores))

fuentes_velocidad = [
    data.get("fuente_velocidad")
    for _, _, _, data in G_osm.edges(keys=True, data=True)
]

pd.Series(fuentes_velocidad).value_counts()

G_osm existe: True
df_pares existe: True
medidores existe: True
Aristas G_osm: 61843
Pares candidatos: 78514
Medidores: 5072


,count
geojson_inferencia,61804
sin_atributos_geojson,39


In [53]:
edges_con_tiempo = []

for u, v, k, data in G_osm.edges(keys=True, data=True):
    tiempo = data.get("travel_time_s")

    if tiempo is not None and pd.notna(tiempo) and np.isfinite(float(tiempo)):
        edges_con_tiempo.append((u, v, k))

G_osm_tiempo = G_osm.edge_subgraph(edges_con_tiempo).copy()

print("Aristas originales G_osm:", G_osm.number_of_edges())
print("Aristas con tiempo disponible:", G_osm_tiempo.number_of_edges())
print("Nodos con tiempo disponible:", G_osm_tiempo.number_of_nodes())

Aristas originales G_osm: 61843
Aristas con tiempo disponible: 61804
Nodos con tiempo disponible: 31433


In [54]:
def metricas_camino_osm_tiempo(G, path):

    distancia_total_m = 0.0
    tiempo_total_s = 0.0

    for u, v in zip(path[:-1], path[1:]):
        edges = G.get_edge_data(u, v)

        if edges is None:
            return np.nan, np.nan

        aristas_validas = []

        for data in edges.values():
            tiempo = data.get("travel_time_s")

            if tiempo is not None and pd.notna(tiempo) and np.isfinite(float(tiempo)):
                aristas_validas.append(data)

        if len(aristas_validas) == 0:
            return np.nan, np.nan

        mejor_arista = min(
            aristas_validas,
            key=lambda data: float(data.get("travel_time_s"))
        )

        distancia_total_m += float(mejor_arista.get("length", 0))
        tiempo_total_s += float(mejor_arista.get("travel_time_s", 0))

    return distancia_total_m, tiempo_total_s

In [55]:
df_pares_calculo = df_pares.copy()

print("Pares candidatos a calcular:", len(df_pares_calculo))

Pares candidatos a calcular: 78514


In [56]:
aristas_reales_tiempo = []
pares_sin_camino_tiempo = 0

for _, row in tqdm(df_pares_calculo.iterrows(), total=len(df_pares_calculo)):
    id_origen = int(row["id_origen"])
    id_destino = int(row["id_destino"])

    osm_origen = int(row["osm_node_origen"])
    osm_destino = int(row["osm_node_destino"])

    distancia_directa_m = float(row["distancia_directa_m"])

    if osm_origen == osm_destino:
        aristas_reales_tiempo.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": 0.0,
            "tiempo_red_s": 0.0,
            "tiempo_red_min": 0.0,
            "velocidad_media_camino_kmh": np.nan,
            "num_nodos_camino_osm": 1,
            "path_osm": str([osm_origen]),
            "tipo_conexion": "mismo_nodo_osm"
        })
        continue

    try:
        path = nx.shortest_path(
            G_osm_tiempo,
            source=osm_origen,
            target=osm_destino,
            weight="travel_time_s"
        )

        path = [int(n) for n in path]

        distancia_red_m, tiempo_red_s = metricas_camino_osm_tiempo(
            G_osm_tiempo,
            path
        )

        if pd.isna(tiempo_red_s) or tiempo_red_s <= 0:
            pares_sin_camino_tiempo += 1
            continue

        velocidad_media_camino_kmh = distancia_red_m / tiempo_red_s * 3.6

        aristas_reales_tiempo.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": distancia_red_m,
            "tiempo_red_s": tiempo_red_s,
            "tiempo_red_min": tiempo_red_s / 60,
            "velocidad_media_camino_kmh": velocidad_media_camino_kmh,
            "num_nodos_camino_osm": len(path),
            "path_osm": str(path),
            "tipo_conexion": "camino_osm_con_tiempo"
        })

    except (nx.NetworkXNoPath, nx.NodeNotFound):
        pares_sin_camino_tiempo += 1
        continue

df_aristas_reales_tiempo = pd.DataFrame(aristas_reales_tiempo).drop_duplicates()

print("Pares candidatos calculados:", len(df_pares_calculo))
print("Aristas reales con tiempo:", len(df_aristas_reales_tiempo))
print("Pares sin camino con tiempo:", pares_sin_camino_tiempo)

df_aristas_reales_tiempo.head()

100%|██████████| 78514/78514 [00:27<00:00, 2873.68it/s]


Pares candidatos calculados: 78514
Aristas reales con tiempo: 78007
Pares sin camino con tiempo: 507


,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tiempo_red_s,tiempo_red_min,velocidad_media_camino_kmh,num_nodos_camino_osm,path_osm,tipo_conexion
0,6835,6833,32636471,32636471,20.144904,0.000000,0.000000,0.000000,NaN,1,[32636471],mismo_nodo_osm
1,6835,6836,32636471,315261895,91.777636,6655.017406,425.102868,7.085048,56.358271,14,"[32636471, 299293137, 21702042, 315518393, 142...",camino_osm_con_tiempo
2,6835,6837,32636471,315264896,94.127890,2248.920678,150.764871,2.512748,53.700271,20,"[32636471, 315266287, 1478929267, 1175412866, ...",camino_osm_con_tiempo
3,6835,6827,32636471,315261895,110.782583,6655.017406,425.102868,7.085048,56.358271,14,"[32636471, 299293137, 21702042, 315518393, 142...",camino_osm_con_tiempo
4,6835,1042,32636471,315265031,116.942125,2583.076553,190.863576,3.181060,48.721059,21,"[32636471, 315266287, 1478929267, 1175412866, ...",camino_osm_con_tiempo


In [57]:
df_aristas_reales_tiempo["factor_rodeo"] = np.where(
    df_aristas_reales_tiempo["distancia_directa_m"] > 0,
    df_aristas_reales_tiempo["distancia_red_m"] / df_aristas_reales_tiempo["distancia_directa_m"],
    np.nan
)

df_aristas_reales_tiempo.loc[
    df_aristas_reales_tiempo["tipo_conexion"] == "mismo_nodo_osm",
    "factor_rodeo"
] = 0

df_aristas_reales_tiempo[
    [
        "distancia_red_m",
        "tiempo_red_s",
        "tiempo_red_min",
        "velocidad_media_camino_kmh",
        "factor_rodeo"
    ]
].describe()

,distancia_red_m,tiempo_red_s,tiempo_red_min,velocidad_media_camino_kmh,factor_rodeo
count,78007.000000,78007.000000,78007.000000,74395.000000,78007.000000
mean,775.783529,60.482971,1.008050,43.467800,3.816856
std,1185.460110,72.563616,1.209394,10.552636,11.472775
min,0.000000,0.000000,0.000000,20.000000,0.000000
25%,238.334734,20.516094,0.341935,34.994116,1.146988
50%,446.007888,39.530694,0.658845,43.549163,1.622164
75%,801.382160,71.587144,1.193119,50.000000,2.938770
max,16339.683383,897.125526,14.952092,90.000000,736.567296


In [58]:
#comprobación de aristas recíprocas en el grafo

df_dir = df_aristas_reales_tiempo.copy()

pares_dirigidos = set(
    zip(df_dir["id_origen"], df_dir["id_destino"])
)

df_dir["tiene_reciproca"] = df_dir.apply(
    lambda row: (row["id_destino"], row["id_origen"]) in pares_dirigidos,
    axis=1
)

porcentaje_reciprocas = df_dir["tiene_reciproca"].mean() * 100

print("Aristas totales:", len(df_dir))
print("Aristas con recíproca:", df_dir["tiene_reciproca"].sum())
print("Porcentaje con recíproca:", round(porcentaje_reciprocas, 2), "%")

Aristas totales: 78007
Aristas con recíproca: 77584
Porcentaje con recíproca: 99.46 %


In [59]:
#calidad de aristas con tiempo

umbral_rodeo_alto = df_aristas_reales_tiempo["factor_rodeo"].quantile(0.95)
umbral_distancia_alta = df_aristas_reales_tiempo["distancia_red_m"].quantile(0.95)

print("Umbral rodeo alto:", round(umbral_rodeo_alto, 2))
print("Umbral distancia alta:", round(umbral_distancia_alta, 2), "m")

df_aristas_reales_tiempo["calidad_arista"] = "ok"

cond_rodeo = df_aristas_reales_tiempo["factor_rodeo"] > umbral_rodeo_alto
cond_distancia = df_aristas_reales_tiempo["distancia_red_m"] > umbral_distancia_alta

df_aristas_reales_tiempo.loc[
    cond_rodeo,
    "calidad_arista"
] = "revisar_rodeo_alto"

df_aristas_reales_tiempo.loc[
    cond_distancia,
    "calidad_arista"
] = "revisar_distancia_alta"

df_aristas_reales_tiempo.loc[
    cond_rodeo & cond_distancia,
    "calidad_arista"
] = "revisar_rodeo_y_distancia_alta"

print("Resumen calidad de aristas:")
print(df_aristas_reales_tiempo["calidad_arista"].value_counts())

Umbral rodeo alto: 12.17
Umbral distancia alta: 2859.98 m
Resumen calidad de aristas:
calidad_arista
ok                                72741
revisar_rodeo_y_distancia_alta     2534
revisar_rodeo_alto                 1367
revisar_distancia_alta             1365
Name: count, dtype: int64


In [60]:
# Guardar aristas marcadas para revisión manual

df_aristas_revisar = df_aristas_reales_tiempo[
    df_aristas_reales_tiempo["calidad_arista"] != "ok"
].copy()

ruta_aristas_revisar = PROCESSED_DIR / "aristas_revisar_manual_tiempo.csv"

df_aristas_revisar.to_csv(
    ruta_aristas_revisar,
    index=False,
    encoding="utf-8"
)

print("Aristas para revisar manualmente:", len(df_aristas_revisar))
print("Guardado en:", ruta_aristas_revisar)

df_aristas_revisar.head()

Aristas para revisar manualmente: 5266
Guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/processed/aristas_revisar_manual_tiempo.csv


,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tiempo_red_s,tiempo_red_min,velocidad_media_camino_kmh,num_nodos_camino_osm,path_osm,tipo_conexion,factor_rodeo,calidad_arista
1,6835,6836,32636471,315261895,91.777636,6655.017406,425.102868,7.085048,56.358271,14,"[32636471, 299293137, 21702042, 315518393, 142...",camino_osm_con_tiempo,72.512408,revisar_rodeo_y_distancia_alta
2,6835,6837,32636471,315264896,94.127890,2248.920678,150.764871,2.512748,53.700271,20,"[32636471, 315266287, 1478929267, 1175412866, ...",camino_osm_con_tiempo,23.892182,revisar_rodeo_alto
3,6835,6827,32636471,315261895,110.782583,6655.017406,425.102868,7.085048,56.358271,14,"[32636471, 299293137, 21702042, 315518393, 142...",camino_osm_con_tiempo,60.072777,revisar_rodeo_y_distancia_alta
4,6835,1042,32636471,315265031,116.942125,2583.076553,190.863576,3.181060,48.721059,21,"[32636471, 315266287, 1478929267, 1175412866, ...",camino_osm_con_tiempo,22.088504,revisar_rodeo_alto
5,6835,6838,32636471,315264896,117.428537,2248.920678,150.764871,2.512748,53.700271,20,"[32636471, 315266287, 1478929267, 1175412866, ...",camino_osm_con_tiempo,19.151398,revisar_rodeo_alto


In [61]:
import ast
from shapely.geometry import LineString, MultiLineString

def color_por_velocidad(velocidad):
    if pd.isna(velocidad) or velocidad < 0:
        return "gray"
    elif velocidad <= 20:
        return "darkred"
    elif velocidad <= 30:
        return "red"
    elif velocidad <= 50:
        return "orange"
    elif velocidad <= 70:
        return "green"
    else:
        return "blue"


def mejor_arista_por_tiempo(G, u, v):
    edges = G.get_edge_data(u, v)

    if edges is None:
        return None

    aristas_validas = []

    for data in edges.values():
        tiempo = data.get("travel_time_s")

        if tiempo is not None and pd.notna(tiempo):
            aristas_validas.append(data)

    if len(aristas_validas) == 0:
        return None

    return min(aristas_validas, key=lambda d: float(d.get("travel_time_s")))


def coords_arista_osm(G, u, v):
    data = mejor_arista_por_tiempo(G, u, v)

    if data is None:
        return []

    geom = data.get("geometry")

    if isinstance(geom, LineString):
        return [[lat, lon] for lon, lat in geom.coords]

    elif isinstance(geom, MultiLineString):
        coords = []
        for linea in geom.geoms:
            coords.extend([[lat, lon] for lon, lat in linea.coords])
        return coords

    else:
        lat_u = G.nodes[u]["y"]
        lon_u = G.nodes[u]["x"]
        lat_v = G.nodes[v]["y"]
        lon_v = G.nodes[v]["x"]

        return [[lat_u, lon_u], [lat_v, lon_v]]


def coords_camino_osm(G, path):
    coords_totales = []

    for u, v in zip(path[:-1], path[1:]):
        coords = coords_arista_osm(G, u, v)

        if len(coords) == 0:
            continue

        if len(coords_totales) == 0:
            coords_totales.extend(coords)
        else:
            coords_totales.extend(coords[1:])

    return coords_totales

In [62]:
import folium
from tqdm import tqdm

centro_madrid = [
    medidores["latitud"].mean(),
    medidores["longitud"].mean()
]

mapa_caminos_reales = folium.Map(
    location=centro_madrid,
    zoom_start=12,
    tiles="cartodbpositron",
    prefer_canvas=True
)

df_mapa = df_aristas_reales_tiempo.copy()

print("Aristas a pintar:", len(df_mapa))

rutas_pintadas = 0
rutas_sin_path = 0


for _, row in tqdm(df_mapa.iterrows(), total=len(df_mapa)):
    try:
        if pd.notna(row["path_osm"]):
            path = ast.literal_eval(row["path_osm"])
        else:
            path = nx.shortest_path(
                G_osm_tiempo,
                source=int(row["osm_node_origen"]),
                target=int(row["osm_node_destino"]),
                weight="travel_time_s"
            )

        path = [int(n) for n in path]

        if len(path) <= 1:
            continue

        coords = coords_camino_osm(G_osm_tiempo, path)

        if len(coords) == 0:
            rutas_sin_path += 1
            continue

        velocidad = row["velocidad_media_camino_kmh"]
        tiempo_min = row["tiempo_red_min"]
        distancia_m = row["distancia_red_m"]

        popup = (
            f"<b>Camino real OSM</b><br>"
            f"ID origen: {int(row['id_origen'])}<br>"
            f"ID destino: {int(row['id_destino'])}<br>"
            f"Distancia red: {distancia_m:.1f} m<br>"
            f"Tiempo: {tiempo_min:.2f} min<br>"
            f"Velocidad media: {velocidad:.1f} km/h<br>"
            f"Nodos OSM camino: {int(row['num_nodos_camino_osm'])}"
        )

        folium.PolyLine(
            locations=coords,
            color=color_por_velocidad(velocidad),
            weight=2,
            opacity=0.55,
            popup=popup
        ).add_to(mapa_caminos_reales)

        rutas_pintadas += 1

    except Exception:
        rutas_sin_path += 1
        continue

nodos_en_mapa = set(df_mapa["id_origen"]).union(set(df_mapa["id_destino"]))

print("Nodos medidores a pintar:", len(nodos_en_mapa))

medidores_mapa = medidores[medidores["id"].isin(nodos_en_mapa)].copy()

for _, row in tqdm(medidores_mapa.iterrows(), total=len(medidores_mapa)):
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=3,
        color="black",
        fill=True,
        fill_color="black",
        fill_opacity=0.9,
        popup=(
            f"<b>Medidor</b><br>"
            f"ID: {int(row['id'])}<br>"
            f"Nombre: {row['nombre']}<br>"
            f"OSM node: {int(row['osm_node'])}"
        )
    ).add_to(mapa_caminos_reales)


leyenda_html = """
<div style="
position: fixed;
bottom: 40px; left: 40px; width: 270px; height: 210px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding: 10px;
">
<b>Grafo de medidores</b><br>
<span style="color:black;">●</span> Nodo medidor<br><br>

<b>Velocidad media del camino OSM</b><br>
<span style="color:darkred;">━━</span> ≤ 20 km/h<br>
<span style="color:red;">━━</span> 21 - 30 km/h<br>
<span style="color:orange;">━━</span> 31 - 50 km/h<br>
<span style="color:green;">━━</span> 51 - 70 km/h<br>
<span style="color:blue;">━━</span> > 70 km/h<br>
<span style="color:gray;">━━</span> sin dato<br>
</div>
"""

mapa_caminos_reales.get_root().html.add_child(folium.Element(leyenda_html))

ruta_mapa_caminos_reales = PROCESSED_DIR / "mapa_caminos_reales_osm_medidores_completo.html"

mapa_caminos_reales.save(ruta_mapa_caminos_reales)

print("Rutas pintadas:", rutas_pintadas)
print("Rutas no pintadas:", rutas_sin_path)
print("Mapa guardado en:", ruta_mapa_caminos_reales)
print("Existe:", ruta_mapa_caminos_reales.exists())



Aristas a pintar: 78007


100%|██████████| 78007/78007 [00:36<00:00, 2154.72it/s]


Nodos medidores a pintar: 5069


100%|██████████| 5069/5069 [00:00<00:00, 10210.15it/s]


Rutas pintadas: 74395
Rutas no pintadas: 0
Mapa guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/processed/mapa_caminos_reales_osm_medidores_completo.html
Existe: True


## Revisión de aristas OSM sin atributos de velocidad

In [63]:
print("Aristas OSM con atributos del GeoJSON:", aristas_con_atributos)
print("Aristas OSM sin atributos del GeoJSON:", aristas_sin_atributos)

Aristas OSM con atributos del GeoJSON: 61804
Aristas OSM sin atributos del GeoJSON: 39


In [64]:
# Revisión de aristas de G_osm sin atributos del GeoJSON de velocidades

aristas_sin_atributos_revision = []

set_geojson = set(
    zip(
        gdf_velocidades_validas["u"].astype(int),
        gdf_velocidades_validas["v"].astype(int)
    )
)

for u, v, k, data in G_osm.edges(keys=True, data=True):
    if data.get("fuente_velocidad") == "sin_atributos_geojson":
        u_int = int(u)
        v_int = int(v)

        existe_directo = (u_int, v_int) in set_geojson
        existe_inverso = (v_int, u_int) in set_geojson

        aristas_sin_atributos_revision.append({
            "u": u_int,
            "v": v_int,
            "key": int(k),
            "osmid": str(data.get("osmid")),
            "name": str(data.get("name")),
            "highway": str(data.get("highway")),
            "length": data.get("length"),
            "existe_en_geojson_directo": existe_directo,
            "existe_en_geojson_inverso": existe_inverso,
            "diagnostico": "existe_inverso" if existe_inverso else "no_existe_en_geojson"
        })

df_aristas_sin_atributos_revision = pd.DataFrame(aristas_sin_atributos_revision)

total_aristas_osm = G_osm.number_of_edges()
total_sin_atributos = len(df_aristas_sin_atributos_revision)
porcentaje_sin_atributos = total_sin_atributos / total_aristas_osm * 100

print("Aristas totales en G_osm:", total_aristas_osm)
print("Aristas sin atributos del GeoJSON:", total_sin_atributos)
print("Porcentaje sin atributos:", round(porcentaje_sin_atributos, 4), "%")

if total_sin_atributos > 0:
    print("\nDiagnóstico:")
    print(df_aristas_sin_atributos_revision["diagnostico"].value_counts())

df_aristas_sin_atributos_revision.head(50)

Aristas totales en G_osm: 61843
Aristas sin atributos del GeoJSON: 39
Porcentaje sin atributos: 0.0631 %

Diagnóstico:
diagnostico
no_existe_en_geojson    38
existe_inverso           1
Name: count, dtype: int64


,u,v,key,osmid,name,highway,length,existe_en_geojson_directo,existe_en_geojson_inverso,diagnostico
0,21994228,13949072791,0,147157053,Calle República Checa,residential,81.495059,False,False,no_existe_en_geojson
1,25906978,1505542782,0,"[1453554752, 75336312]",Paseo de la Castellana,"['trunk', 'trunk_link']",150.981486,False,False,no_existe_en_geojson
2,25908512,13953905230,0,71045417,Paseo de la Castellana,secondary,14.343175,False,False,no_existe_en_geojson
3,31031060,248001162,0,208914108,Calle de Pedro Laborde,residential,69.000886,False,False,no_existe_en_geojson
4,137450258,13953905202,0,1531015358,None,busway,187.648253,False,False,no_existe_en_geojson
5,137452400,13953905201,0,298699352,Plaza de Andrés Manjón,residential,2.467973,False,False,no_existe_en_geojson
6,137485778,13953905202,0,14306536,Plaza de Andrés Manjón,residential,53.911366,False,False,no_existe_en_geojson
7,307644015,316637757,0,28016640,Calle Sierra Faladora,residential,107.627581,False,False,no_existe_en_geojson
8,316637698,317567808,0,28798587,Calle Pedro Callejo,residential,88.226593,False,False,no_existe_en_geojson
9,317567786,1250622590,0,93519938,Calle Adra,residential,98.136112,False,False,no_existe_en_geojson


CONCLUSIÓN

Tras revisar las aristas sin atributos de velocidad, se observa que solo 39 aristas de unas 61.845 no tienen correspondencia con el GeoJSON de velocidades.

Esto representa un porcentaje muy pequeño del grafo, por lo que no afecta de forma relevante al resultado final.

Estas aristas se dejan marcadas como dato faltante y el proceso continúa usando únicamente las aristas que sí tienen velocidad y tiempo de recorrido.

# Bloque 2 (cierre): Reconstrucción de G_medidores con tiempo y decisión de direccionalidad

In [65]:
# Reconstrucción de G_medidores: con tiempo de viaje y como grafo NO dirigido
# (justificado por el 99.46% de aristas con recíproca calculado antes)

G_medidores = nx.Graph()

# Añadir todos los medidores reales como nodos
for _, row in medidores.iterrows():
    G_medidores.add_node(
        int(row["id"]),
        nombre=row["nombre"],
        latitud=float(row["latitud"]),
        longitud=float(row["longitud"]),
        osm_node=int(row["osm_node"]),
        distancia_osm_node_m=float(row["distancia_osm_node_m"])
    )

# Añadir aristas reales calculadas sobre OSM, con tiempo y calidad ya correctas
for _, row in df_aristas_reales_tiempo.iterrows():
    G_medidores.add_edge(
        int(row["id_origen"]),
        int(row["id_destino"]),
        distancia_directa_m=float(row["distancia_directa_m"]),
        distancia_red_m=float(row["distancia_red_m"]),
        tiempo_red_s=float(row["tiempo_red_s"]),
        tiempo_red_min=float(row["tiempo_red_min"]),
        factor_rodeo=float(row["factor_rodeo"]),
        weight=float(row["tiempo_red_s"]),
        tipo_conexion=row["tipo_conexion"],
        calidad_arista=row["calidad_arista"]
    )

print("Nodos:", G_medidores.number_of_nodes())
print("Aristas:", G_medidores.number_of_edges())

Nodos: 5072
Aristas: 39215


In [66]:
componentes = list(nx.connected_components(G_medidores))
componentes_ordenados = sorted(componentes, key=len, reverse=True)

print("Número de componentes:", len(componentes_ordenados))
for i, c in enumerate(componentes_ordenados[:10], start=1):
    print(f"Componente {i}: {len(c)} nodos")

Número de componentes: 26
Componente 1: 4787 nodos
Componente 2: 64 nodos
Componente 3: 43 nodos
Componente 4: 39 nodos
Componente 5: 25 nodos
Componente 6: 17 nodos
Componente 7: 13 nodos
Componente 8: 12 nodos
Componente 9: 11 nodos
Componente 10: 8 nodos


In [67]:
import networkx as nx
import pandas as pd
import numpy as np

def limpiar_valor_graphml(valor):
    if valor is None:
        return ""
    if isinstance(valor, float) and pd.isna(valor):
        return -1.0
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating,)):
        return float(valor)
    if isinstance(valor, (np.bool_,)):
        return bool(valor)
    if isinstance(valor, (list, tuple, set, dict)):
        return str(valor)
    if not isinstance(valor, (str, int, float, bool)):
        return str(valor)
    return valor

G_medidores_limpio = nx.Graph()

for nodo, data in G_medidores.nodes(data=True):
    G_medidores_limpio.add_node(
        nodo,
        **{k: limpiar_valor_graphml(v) for k, v in data.items()}
    )

for u, v, data in G_medidores.edges(data=True):
    G_medidores_limpio.add_edge(
        u,
        v,
        **{k: limpiar_valor_graphml(v) for k, v in data.items()}
    )

ruta_grafo_medidores_final = GRAPH_DIR / "grafo_medidores_madrid_tiempo_no_dirigido.graphml"

nx.write_graphml(
    G_medidores_limpio,
    ruta_grafo_medidores_final
)

print("Grafo guardado en:", ruta_grafo_medidores_final)
print("Existe:", ruta_grafo_medidores_final.exists())
print("Tamaño MB:", round(ruta_grafo_medidores_final.stat().st_size / (1024 * 1024), 2))

Grafo guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/graphs/grafo_medidores_madrid_tiempo_no_dirigido.graphml
Existe: True
Tamaño MB: 15.16


In [68]:
G_prueba = nx.read_graphml(ruta_grafo_medidores_final)

print("Grafo cargado correctamente")
print("Nodos:", G_prueba.number_of_nodes())
print("Aristas:", G_prueba.number_of_edges())

Grafo cargado correctamente
Nodos: 5072
Aristas: 39215


In [69]:
print(df_aristas_reales_tiempo[["distancia_directa_m", "distancia_red_m", "tiempo_red_s", "tiempo_red_min", "factor_rodeo", "tipo_conexion", "calidad_arista"]].isna().sum())

distancia_directa_m    0
distancia_red_m        0
tiempo_red_s           0
tiempo_red_min         0
factor_rodeo           0
tipo_conexion          0
calidad_arista         0
dtype: int64


In [70]:
print(medidores[["nombre", "latitud", "longitud", "osm_node", "distancia_osm_node_m"]].isna().sum())

nombre                  23
latitud                  0
longitud                 0
osm_node                 0
distancia_osm_node_m     0
dtype: int64


In [71]:
from pathlib import Path

archivos_en_proyecto = list(BASE_DIR.rglob("*"))
print("Total de archivos/carpetas:", len(archivos_en_proyecto))
for f in archivos_en_proyecto:
    print(f.relative_to(BASE_DIR))

Total de archivos/carpetas: 28
data
mapa_velocidades
data/raw
data/processed
data/graphs
mapa_velocidades/velocidades_madrid_final.geojson
data/raw/202468-292-intensidad-trafico.zip
data/raw/pmed_ubicacion
data/raw/208627-157-transporte-ptomedida-historico.zip
data/raw/historico_mayo_2026
data/processed/mapa_grafo_medidores.html
data/processed/aristas_reales_medidores_con_tiempo.csv
data/processed/analisis_componentes.csv
data/processed/mapa_componentes.html
data/processed/aristas_revisar_manual_tiempo.csv
data/processed/mapa_caminos_reales_osm_medidores_completo.html
data/processed/mediana_historica_velocidad_por_medidor_hora.csv
data/processed/velocidad_respaldo_por_medidor_hora.csv
data/graphs/grafo_medidores_madrid_tiempo.graphml
data/graphs/red_osm_madrid_base.graphml
data/graphs/grafo_medidores_madrid_tiempo_no_dirigido.graphml
data/graphs/Copy_of_grafo_puntos_medidores.ipynb
data/raw/pmed_ubicacion/pmed_ubicacion_05-2026.cpg
data/raw/pmed_ubicacion/pmed_ubicacion_05-2026.prj
dat

In [72]:
import requests

url_historico = "https://datos.madrid.es/dataset/208627-0-transporte-ptomedida-historico/resource/208627-157-transporte-ptomedida-historico/download/208627-157-transporte-ptomedida-historico.zip"

ruta_zip_historico = RAW_DIR / "208627-157-transporte-ptomedida-historico.zip"

print("Descargando histórico de mayo 2026...")
respuesta = requests.get(url_historico, timeout=120)
respuesta.raise_for_status()

with open(ruta_zip_historico, "wb") as f:
    f.write(respuesta.content)

print("Descargado en:", ruta_zip_historico)
print("Tamaño:", ruta_zip_historico.stat().st_size / (1024*1024), "MB")

Descargando histórico de mayo 2026...
Descargado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/208627-157-transporte-ptomedida-historico.zip
Tamaño: 92.71568393707275 MB


In [73]:
import zipfile

carpeta_historico = RAW_DIR / "historico_mayo_2026"
carpeta_historico.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ruta_zip_historico, "r") as zip_ref:
    print("Contenido del ZIP:")
    for nombre in zip_ref.namelist():
        print(nombre)
    zip_ref.extractall(carpeta_historico)

print("\nArchivos extraídos en:", carpeta_historico)
for f in carpeta_historico.iterdir():
    print(f.name, "-", f.stat().st_size / (1024*1024), "MB")

Contenido del ZIP:
05-2026.csv

Archivos extraídos en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/raw/historico_mayo_2026
05-2026.csv - 783.1925277709961 MB


In [74]:
ruta_csv_historico = carpeta_historico / "05-2026.csv"

# Leer solo las primeras 5 filas para ver la estructura, sin cargar todo el archivo
muestra = pd.read_csv(ruta_csv_historico, sep=";", nrows=5)
print("Columnas:", muestra.columns.tolist())
muestra

Columnas: ['id', 'fecha', 'tipo_elem', 'intensidad', 'ocupacion', 'carga', 'vmed', 'error', 'periodo_integracion']


,id,fecha,tipo_elem,intensidad,ocupacion,carga,vmed,error,periodo_integracion
0,1001,2026-05-01 00:00:00,C30,0,0,0,0,N,5
1,1001,2026-05-01 00:15:00,C30,0,0,0,0,N,5
2,1001,2026-05-01 00:30:00,C30,0,0,0,0,N,5
3,1001,2026-05-01 00:45:00,C30,0,0,0,0,N,5
4,1001,2026-05-01 01:00:00,C30,0,0,0,0,N,5


In [75]:
import pandas as pd

columnas_necesarias = ["id", "fecha", "vmed", "error"]

print("Cargando histórico completo (puede tardar un poco)...")
df_historico = pd.read_csv(
    ruta_csv_historico,
    sep=";",
    usecols=columnas_necesarias,
    dtype={"id": "int32", "vmed": "float32", "error": "category"},
    parse_dates=["fecha"]
)

print("Filas cargadas:", len(df_historico))
print("Memoria usada (MB):", df_historico.memory_usage(deep=True).sum() / (1024*1024))
df_historico.head()

Cargando histórico completo (puede tardar un poco)...
Filas cargadas: 13775364
Memoria usada (MB): 223.3327579498291


,id,fecha,vmed,error
0,1001,2026-05-01 00:00:00,0.0,N
1,1001,2026-05-01 00:15:00,0.0,N
2,1001,2026-05-01 00:30:00,0.0,N
3,1001,2026-05-01 00:45:00,0.0,N
4,1001,2026-05-01 01:00:00,0.0,N


In [76]:
print("Valores únicos de 'error':", df_historico["error"].unique().tolist())

df_historico_valido = df_historico[df_historico["error"] == "N"].copy()

print("Filas totales:", len(df_historico))
print("Filas válidas (error = N):", len(df_historico_valido))
print("Filas descartadas:", len(df_historico) - len(df_historico_valido))

Valores únicos de 'error': ['N', nan]
Filas totales: 13775364
Filas válidas (error = N): 13690926
Filas descartadas: 84438


In [77]:
df_historico_valido["hora"] = df_historico_valido["fecha"].dt.hour

print(df_historico_valido["hora"].value_counts().sort_index())

hora
0     572800
1     564727
2     555137
3     545863
4     544768
5     553416
6     556275
7     566227
8     573437
9     576371
10    576407
11    574232
12    574280
13    576814
14    577770
15    578548
16    578429
17    578617
18    578669
19    579063
20    578372
21    577902
22    576950
23    575852
Name: count, dtype: int64


In [78]:
mediana_historica = (
    df_historico_valido
    .groupby(["id", "hora"])["vmed"]
    .median()
    .reset_index()
    .rename(columns={"vmed": "velocidad_mediana_kmh"})
)

print("Filas resultado (medidor x hora):", len(mediana_historica))
mediana_historica.head(10)

Filas resultado (medidor x hora): 114029


,id,hora,velocidad_mediana_kmh
0,1001,0,0.0
1,1001,1,0.0
2,1001,2,0.0
3,1001,3,0.0
4,1001,4,0.0
5,1001,5,0.0
6,1001,6,0.0
7,1001,7,0.0
8,1001,8,0.0
9,1001,9,0.0


In [79]:
porcentaje_cero = (mediana_historica["velocidad_mediana_kmh"] == 0).mean() * 100
print(f"Porcentaje de filas con mediana 0: {porcentaje_cero:.2f}%")

# Comprobar si el medidor 1001 tiene SIEMPRE 0, o solo en algunas horas
print(df_historico_valido[df_historico_valido["id"] == 1001]["vmed"].describe())

Porcentaje de filas con mediana 0: 92.03%
count    2976.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: vmed, dtype: float64


In [80]:
# Para cada medidor, comprobar si TIENE algún registro con velocidad mayor que 0
velocidad_max_por_medidor = df_historico_valido.groupby("id")["vmed"].max()

medidores_con_velocidad_real = velocidad_max_por_medidor[velocidad_max_por_medidor > 0].index
medidores_sin_velocidad_real = velocidad_max_por_medidor[velocidad_max_por_medidor == 0].index

print("Medidores con velocidad real alguna vez:", len(medidores_con_velocidad_real))
print("Medidores SIN velocidad real (siempre 0):", len(medidores_sin_velocidad_real))
print("Total medidores en el histórico:", len(velocidad_max_por_medidor))

Medidores con velocidad real alguna vez: 377
Medidores SIN velocidad real (siempre 0): 4403
Total medidores en el histórico: 4786


In [81]:
df_historico_real = df_historico_valido[
    df_historico_valido["id"].isin(medidores_con_velocidad_real)
].copy()

mediana_historica_real = (
    df_historico_real
    .groupby(["id", "hora"])["vmed"]
    .median()
    .reset_index()
    .rename(columns={"vmed": "velocidad_mediana_kmh"})
)

print("Filas resultado (solo medidores con dato real):", len(mediana_historica_real))
print("Medidores distintos:", mediana_historica_real["id"].nunique())
mediana_historica_real.head(10)

Filas resultado (solo medidores con dato real): 9006
Medidores distintos: 377


,id,hora,velocidad_mediana_kmh
0,1006,0,60.0
1,1006,1,60.0
2,1006,2,60.0
3,1006,3,60.0
4,1006,4,60.0
5,1006,5,60.0
6,1006,6,61.0
7,1006,7,63.0
8,1006,8,62.0
9,1006,9,61.0


In [82]:
print(mediana_historica_real["velocidad_mediana_kmh"].describe())

count    8984.000000
mean       66.810661
std        18.510590
min         0.000000
25%        57.000000
50%        68.000000
75%        80.000000
max       125.500000
Name: velocidad_mediana_kmh, dtype: float64


In [83]:
print("Filas con valor nulo:", mediana_historica_real["velocidad_mediana_kmh"].isna().sum())

casos_cero = mediana_historica_real[mediana_historica_real["velocidad_mediana_kmh"] == 0]
print("\nCasos con mediana 0:")
print(casos_cero)

Filas con valor nulo: 22

Casos con mediana 0:
         id  hora  velocidad_mediana_kmh
624    1045     0                    0.0
625    1045     1                    0.0
626    1045     2                    0.0
627    1045     3                    0.0
628    1045     4                    0.0
629    1045     5                    0.0
630    1045     6                    0.0
631    1045     7                    0.0
632    1045     8                    0.0
633    1045     9                    0.0
634    1045    10                    0.0
635    1045    11                    0.0
636    1045    12                    0.0
637    1045    13                    0.0
638    1045    14                    0.0
639    1045    15                    0.0
640    1045    16                    0.0
641    1045    17                    0.0
642    1045    18                    0.0
643    1045    19                    0.0
644    1045    20                    0.0
645    1045    21                    0.0
646    104

In [84]:
# Para cada medidor, calcular qué porcentaje de sus mediciones tienen velocidad > 0
porcentaje_con_velocidad = (
    df_historico_valido.groupby("id")["vmed"]
    .apply(lambda x: (x > 0).mean() * 100)
)

print(porcentaje_con_velocidad.describe())
print("\nMedidores con menos del 50% de mediciones con velocidad >0:")
print(porcentaje_con_velocidad[(porcentaje_con_velocidad > 0) & (porcentaje_con_velocidad < 50)].sort_values())

count    4786.000000
mean        7.738742
std        26.592192
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       100.000000
Name: vmed, dtype: float64

Medidores con menos del 50% de mediciones con velocidad >0:
id
11369     0.237248
1045      0.739247
11498    16.901882
6807     33.333333
Name: vmed, dtype: float64


In [85]:
medidores_fiables = porcentaje_con_velocidad[porcentaje_con_velocidad >= 50].index

print("Medidores fiables para mediana histórica:", len(medidores_fiables))

df_historico_fiable = df_historico_valido[
    df_historico_valido["id"].isin(medidores_fiables)
].copy()

mediana_historica_final = (
    df_historico_fiable
    .groupby(["id", "hora"])["vmed"]
    .median()
    .reset_index()
    .rename(columns={"vmed": "velocidad_mediana_kmh"})
)

print("Filas resultado:", len(mediana_historica_final))
print(mediana_historica_final["velocidad_mediana_kmh"].describe())

Medidores fiables para mediana histórica: 373
Filas resultado: 8910
count    8910.000000
mean       67.357910
std        17.581503
min         0.000000
25%        57.000000
50%        68.000000
75%        80.000000
max       125.500000
Name: velocidad_mediana_kmh, dtype: float64


In [86]:
ruta_mediana_historica = PROCESSED_DIR / "mediana_historica_velocidad_por_medidor_hora.csv"

mediana_historica_final.to_csv(ruta_mediana_historica, index=False)

print("Guardado en:", ruta_mediana_historica)
print("Existe:", ruta_mediana_historica.exists())

Guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/processed/mediana_historica_velocidad_por_medidor_hora.csv
Existe: True


In [87]:
# Para cada nodo OSM, buscamos alguna arista (tramo) que lo toque, y sacamos su velocidad
# Usamos el GeoJSON de velocidades que ya cargamos al principio (gdf_velocidades_full)

velocidad_por_nodo = {}

for _, row in gdf_velocidades_full.iterrows():
    vel = row["velocidad_kmh"] if "velocidad_kmh" in gdf_velocidades_full.columns else None
    if pd.isna(vel):
        continue
    for nodo in (row["u"], row["v"]):
        if nodo not in velocidad_por_nodo:
            velocidad_por_nodo[nodo] = vel

print("Nodos con velocidad legal asignada:", len(velocidad_por_nodo))

Nodos con velocidad legal asignada: 0


In [88]:
print([v for v in dir() if "velocidad" in v.lower()])

['METODO_AGREGACION_VELOCIDAD', 'VELOCIDADES_DIR', 'agregar_velocidad', 'color_por_velocidad', 'extraer_numeros_velocidad', 'fuentes_velocidad', 'gdf_velocidades', 'gdf_velocidades_full', 'gdf_velocidades_validas', 'medidores_con_velocidad_real', 'medidores_sin_velocidad_real', 'porcentaje_con_velocidad', 'ruta_velocidades', 'velocidad', 'velocidad_max_por_medidor', 'velocidad_media_camino_kmh', 'velocidad_por_nodo']


In [89]:
for nombre_var in ["gdf_velocidades_full", "gdf_velocidades", "gdf_velocidades_validas"]:
    var = eval(nombre_var)
    print(nombre_var, "- columnas:", var.columns.tolist())
    print(nombre_var, "- filas:", len(var))
    print()

gdf_velocidades_full - columnas: ['u', 'v', 'key', 'highway', 'maxspeed', 'es_urbano', 'maxspeed_final', 'geometry']
gdf_velocidades_full - filas: 271232

gdf_velocidades - columnas: ['u', 'v', 'key', 'highway', 'maxspeed', 'es_urbano', 'maxspeed_final', 'geometry', 'velocidad_kmh']
gdf_velocidades - filas: 271232

gdf_velocidades_validas - columnas: ['u', 'v', 'key', 'highway', 'maxspeed', 'es_urbano', 'maxspeed_final', 'geometry', 'velocidad_kmh']
gdf_velocidades_validas - filas: 271232



In [90]:
velocidad_por_nodo = {}

for _, row in gdf_velocidades.iterrows():
    vel = row["velocidad_kmh"]
    if pd.isna(vel):
        continue
    for nodo in (row["u"], row["v"]):
        if nodo not in velocidad_por_nodo:
            velocidad_por_nodo[nodo] = vel

print("Nodos con velocidad legal asignada:", len(velocidad_por_nodo))

Nodos con velocidad legal asignada: 129047


In [91]:
medidores["velocidad_legal_kmh"] = medidores["osm_node"].map(velocidad_por_nodo)

print("Medidores con velocidad legal asignada:", medidores["velocidad_legal_kmh"].notna().sum())
print("Medidores SIN velocidad legal asignada:", medidores["velocidad_legal_kmh"].isna().sum())

medidores[["id", "nombre", "osm_node", "velocidad_legal_kmh"]].head(10)

Medidores con velocidad legal asignada: 5069
Medidores SIN velocidad legal asignada: 3


,id,nombre,osm_node,velocidad_legal_kmh
0,6835,18RA28PM01,32636471,80.0
1,1012,18RA66PM01,315259372,50.0
2,5035,FRUELA N-S,305399713,50.0
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,1672792326,50.0
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,119794656,30.0
5,5585,Costa Rica O-E(Lateral) - Victor de la Serna-P...,98965348,50.0
6,10556,Alcalá - Alegria de Horia-Tampico,2208138736,40.0
7,3435,"Maria Molina, 40 O-E - Velazquez-Nuñez de Balboa",13631291481,50.0
8,3974,Francisco Silvela S-N - Av. America-San Fdo.de...,148889926,30.0
9,3612,"Conde Peñalver, 60 N-S - Padilla-Jose Ortega y...",28096857,50.0


In [92]:
# Convertimos la mediana histórica a un diccionario rápido de consulta: (id, hora) -> velocidad
dict_mediana_historica = {
    (row["id"], row["hora"]): row["velocidad_mediana_kmh"]
    for _, row in mediana_historica_final.iterrows()
}

# Convertimos la velocidad legal a un diccionario rápido: id -> velocidad
dict_velocidad_legal = dict(zip(medidores["id"], medidores["velocidad_legal_kmh"]))


def velocidad_respaldo(id_medidor, hora):
    """
    Devuelve la velocidad de respaldo (km/h) para un medidor en una hora concreta,
    cuando no hay dato de tráfico en tiempo real disponible.
    Prioridad: 1) mediana histórica del propio medidor/hora, 2) velocidad legal del tramo OSM.
    """
    clave = (id_medidor, hora)
    if clave in dict_mediana_historica:
        return dict_mediana_historica[clave], "mediana_historica"
    if id_medidor in dict_velocidad_legal and pd.notna(dict_velocidad_legal[id_medidor]):
        return dict_velocidad_legal[id_medidor], "velocidad_legal"
    return None, "sin_dato"


# Prueba con un par de ejemplos
print(velocidad_respaldo(1006, 8))   # medidor con mediana histórica
print(velocidad_respaldo(6835, 8))   # medidor probablemente sin histórico fiable

(np.float64(62.0), 'mediana_historica')
(np.float64(65.0), 'mediana_historica')


In [93]:
# Coger un medidor cualquiera que NO esté en los 373 fiables
medidor_sin_historico = medidores[~medidores["id"].isin(medidores_fiables)]["id"].iloc[0]

print("Probando con medidor:", medidor_sin_historico)
print(velocidad_respaldo(medidor_sin_historico, 8))

Probando con medidor: 1012
(50.0, 'velocidad_legal')


In [94]:
filas_resumen = []

for id_medidor in medidores["id"]:
    for hora in range(24):
        vel, fuente = velocidad_respaldo(id_medidor, hora)
        filas_resumen.append({
            "id_medidor": id_medidor,
            "hora": hora,
            "velocidad_respaldo_kmh": vel,
            "fuente_respaldo": fuente
        })

df_velocidad_respaldo = pd.DataFrame(filas_resumen)

print("Filas generadas:", len(df_velocidad_respaldo))
print(df_velocidad_respaldo["fuente_respaldo"].value_counts())

ruta_velocidad_respaldo = PROCESSED_DIR / "velocidad_respaldo_por_medidor_hora.csv"
df_velocidad_respaldo.to_csv(ruta_velocidad_respaldo, index=False)
print("\nGuardado en:", ruta_velocidad_respaldo)


Filas generadas: 121728
fuente_respaldo
velocidad_legal      112794
mediana_historica      8910
sin_dato                 24
Name: count, dtype: int64

Guardado en: /content/drive/MyDrive/Movilidad_inteligente_madrid/data/processed/velocidad_respaldo_por_medidor_hora.csv


In [95]:
import osmnx as ox

# Punto de ejemplo: origen del usuario (latitud, longitud)
# (Puerta del Sol, Madrid, como ejemplo)
origen_lat, origen_lon = 40.4168, -3.7038

# Snap a la red OSM, igual que se hizo con los medidores
origen_osm_node = ox.distance.nearest_nodes(G_osm, X=origen_lon, Y=origen_lat)

print("Nodo OSM asignado al origen:", origen_osm_node)

Nodo OSM asignado al origen: 5302264639


In [96]:
# Destino de ejemplo (Plaza de Castilla, Madrid, bastante al norte de Sol)
destino_lat, destino_lon = 40.4660, -3.6889

destino_osm_node = ox.distance.nearest_nodes(G_osm, X=destino_lon, Y=destino_lat)

print("Nodo OSM asignado al destino:", destino_osm_node)

Nodo OSM asignado al destino: 25904522


In [97]:
from math import radians, sin, cos, sqrt, atan2

def distancia_haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))


def medidores_candidatos_cercanos(lat, lon, n=5):
    distancias = medidores.apply(
        lambda row: distancia_haversine_m(lat, lon, row["latitud"], row["longitud"]),
        axis=1
    )
    medidores_con_distancia = medidores.copy()
    medidores_con_distancia["distancia_m"] = distancias
    return medidores_con_distancia.nsmallest(n, "distancia_m")


candidatos_origen = medidores_candidatos_cercanos(origen_lat, origen_lon, n=5)
candidatos_origen[["id", "nombre", "latitud", "longitud", "distancia_m"]]

,id,nombre,latitud,longitud,distancia_m
2148,10608,Carrera San Jeronimo O-E(Pta Sol-Canalejas),40.416646,-3.700742,259.423107
2147,4254,CRUZ S-N(EL POZO -PL.CANALEJAS),40.416323,-3.700694,268.251009
2174,11301,(TACTICO)CCTV CARRERA SAN JERONIMO O-E(PTA SOL...,40.416632,-3.700523,278.068626
3092,10530,SEVILLA N-S (SALIDA PARKING A 5 MTS G.3),40.416852,-3.700348,292.297431
2411,10527,SEVILLA N-S (SALIDA PARKING A 25 MTS G.3),40.416984,-3.700254,300.925447


In [98]:
from math import radians, degrees, sin, cos, atan2


def calcular_bearing(lat1, lon1, lat2, lon2):
    """Calcula el rumbo (bearing) en grados desde el punto 1 hacia el punto 2."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = sin(dlon) * cos(lat2)
    y = cos(lat1) * sin(lat2) - sin(lat1) * cos(lat2) * cos(dlon)
    bearing = degrees(atan2(x, y))
    return (bearing + 360) % 360


def diferencia_angular(angulo1, angulo2):
    """Diferencia mínima entre dos ángulos, entre 0 y 180 grados."""
    diff = abs(angulo1 - angulo2) % 360
    return min(diff, 360 - diff)


# Bearing del origen hacia el destino
bearing_origen_destino = calcular_bearing(origen_lat, origen_lon, destino_lat, destino_lon)
print("Bearing origen -> destino:", round(bearing_origen_destino, 1), "grados")

Bearing origen -> destino: 13.0 grados


In [99]:
distancia_origen_destino = distancia_haversine_m(origen_lat, origen_lon, destino_lat, destino_lon)
print("Distancia origen-destino:", round(distancia_origen_destino), "metros")

usar_filtro_bearing = distancia_origen_destino >= 200
print("¿Se aplica filtro de bearing?", usar_filtro_bearing)

if usar_filtro_bearing:
    candidatos_origen["bearing_hacia_medidor"] = candidatos_origen.apply(
        lambda row: calcular_bearing(origen_lat, origen_lon, row["latitud"], row["longitud"]),
        axis=1
    )
    candidatos_origen["diferencia_angular"] = candidatos_origen["bearing_hacia_medidor"].apply(
        lambda b: diferencia_angular(b, bearing_origen_destino)
    )
    candidatos_origen["pasa_filtro_bearing"] = candidatos_origen["diferencia_angular"] <= 90
else:
    candidatos_origen["pasa_filtro_bearing"] = True

candidatos_origen[["id", "nombre", "distancia_m", "pasa_filtro_bearing"]]

Distancia origen-destino: 5614 metros
¿Se aplica filtro de bearing? True


,id,nombre,distancia_m,pasa_filtro_bearing
2148,10608,Carrera San Jeronimo O-E(Pta Sol-Canalejas),259.423107,True
2147,4254,CRUZ S-N(EL POZO -PL.CANALEJAS),268.251009,True
2174,11301,(TACTICO)CCTV CARRERA SAN JERONIMO O-E(PTA SOL...,278.068626,True
3092,10530,SEVILLA N-S (SALIDA PARKING A 5 MTS G.3),292.297431,True
2411,10527,SEVILLA N-S (SALIDA PARKING A 25 MTS G.3),300.925447,True


In [100]:
candidatos_validos = candidatos_origen[candidatos_origen["pasa_filtro_bearing"]].copy()

tiempos_origen_medidor = []
for _, row in candidatos_validos.iterrows():
    try:
        tiempo_seg = nx.shortest_path_length(
            G_osm_tiempo, origen_osm_node, row["osm_node"], weight="travel_time"
        )
    except nx.NetworkXNoPath:
        tiempo_seg = None
    tiempos_origen_medidor.append(tiempo_seg)

candidatos_validos["tiempo_origen_medidor_s"] = tiempos_origen_medidor

candidatos_validos[["id", "nombre", "distancia_m", "tiempo_origen_medidor_s"]].sort_values("tiempo_origen_medidor_s")

,id,nombre,distancia_m,tiempo_origen_medidor_s
2411,10527,SEVILLA N-S (SALIDA PARKING A 25 MTS G.3),300.925447,16
2148,10608,Carrera San Jeronimo O-E(Pta Sol-Canalejas),259.423107,17
2174,11301,(TACTICO)CCTV CARRERA SAN JERONIMO O-E(PTA SOL...,278.068626,17
2147,4254,CRUZ S-N(EL POZO -PL.CANALEJAS),268.251009,17
3092,10530,SEVILLA N-S (SALIDA PARKING A 5 MTS G.3),292.297431,17


In [101]:
def encontrar_medidor_entrada_optimo(lat_origen, lon_origen, lat_destino, lon_destino, n_candidatos=5):
    """
    Encuentra el medidor óptimo de entrada al grafo desde un punto de origen arbitrario,
    siguiendo la sección 4.3 de la propuesta: snap, candidatos cercanos, filtro de bearing,
    selección por tiempo mínimo.
    """
    origen_node = ox.distance.nearest_nodes(G_osm, X=lon_origen, Y=lat_origen)

    candidatos = medidores_candidatos_cercanos(lat_origen, lon_origen, n=n_candidatos).copy()

    distancia_od = distancia_haversine_m(lat_origen, lon_origen, lat_destino, lon_destino)
    aplicar_filtro = distancia_od >= 200

    if aplicar_filtro:
        bearing_od = calcular_bearing(lat_origen, lon_origen, lat_destino, lon_destino)
        candidatos["diferencia_angular"] = candidatos.apply(
            lambda row: diferencia_angular(
                calcular_bearing(lat_origen, lon_origen, row["latitud"], row["longitud"]),
                bearing_od
            ),
            axis=1
        )
        candidatos = candidatos[candidatos["diferencia_angular"] <= 90]

    if len(candidatos) == 0:
        return None

    tiempos = []
    for _, row in candidatos.iterrows():
        try:
            t = nx.shortest_path_length(G_osm_tiempo, origen_node, row["osm_node"], weight="travel_time")
        except nx.NetworkXNoPath:
            t = None
        tiempos.append(t)

    candidatos["tiempo_origen_medidor_s"] = tiempos
    candidatos_con_tiempo = candidatos.dropna(subset=["tiempo_origen_medidor_s"])

    if len(candidatos_con_tiempo) == 0:
        return None

    return candidatos_con_tiempo.loc[candidatos_con_tiempo["tiempo_origen_medidor_s"].idxmin()]


# Prueba con el mismo ejemplo de antes
resultado = encontrar_medidor_entrada_optimo(origen_lat, origen_lon, destino_lat, destino_lon)
print(resultado[["id", "nombre", "distancia_m", "tiempo_origen_medidor_s"]])

id                                                             10527
nombre                     SEVILLA N-S (SALIDA PARKING A 25 MTS G.3)
distancia_m                                               300.925447
tiempo_origen_medidor_s                                           16
Name: 2411, dtype: object


In [102]:
resultado_destino = encontrar_medidor_entrada_optimo(destino_lat, destino_lon, origen_lat, origen_lon)
print("Medidor de salida (cerca del destino):")
print(resultado_destino[["id", "nombre", "distancia_m", "tiempo_origen_medidor_s"]])

Medidor de salida (cerca del destino):
id                                                                   3421
nombre                     Bravo Murillo E-O - Pl.Castilla-Conde Serrallo
distancia_m                                                    146.656433
tiempo_origen_medidor_s                                                 8
Name: 5065, dtype: object


## Validación de rutas y métricas de precisión

In [104]:
import networkx as nx
import pandas as pd
import numpy as np

# resultado = medidor de entrada cerca del origen
# resultado_destino = medidor de salida cerca del destino

id_medidor_entrada = int(resultado["id"])
id_medidor_salida = int(resultado_destino["id"])

tiempo_origen_medidor_s = float(resultado["tiempo_origen_medidor_s"])
tiempo_medidor_destino_s = float(resultado_destino["tiempo_origen_medidor_s"])

print("Medidor entrada:", id_medidor_entrada)
print("Medidor salida:", id_medidor_salida)
print("Tiempo origen → medidor entrada:", tiempo_origen_medidor_s, "s")
print("Tiempo medidor salida → destino:", tiempo_medidor_destino_s, "s")

# Por si el grafo tiene los IDs como enteros o como texto
def buscar_nodo(G, nodo):
    if nodo in G:
        return nodo
    if str(nodo) in G:
        return str(nodo)
    raise ValueError(f"El nodo {nodo} no está en G_medidores")

nodo_entrada = buscar_nodo(G_medidores, id_medidor_entrada)
nodo_salida = buscar_nodo(G_medidores, id_medidor_salida)

# Ruta por el grafo de medidores
path_medidores = nx.shortest_path(
    G_medidores,
    source=nodo_entrada,
    target=nodo_salida,
    weight="weight"
)

tiempo_grafo_medidores_s = nx.shortest_path_length(
    G_medidores,
    source=nodo_entrada,
    target=nodo_salida,
    weight="weight"
)

tiempo_total_s = (
    tiempo_origen_medidor_s
    + tiempo_grafo_medidores_s
    + tiempo_medidor_destino_s
)

tiempo_total_min = tiempo_total_s / 60

print("Tiempo medidor entrada → medidor salida:", round(tiempo_grafo_medidores_s, 2), "s")
print("Tiempo total estimado:", round(tiempo_total_min, 2), "min")
print("Número de medidores en la ruta:", len(path_medidores))
print("Ruta de medidores:", path_medidores)

Medidor entrada: 10527
Medidor salida: 3421
Tiempo origen → medidor entrada: 16.0 s
Tiempo medidor salida → destino: 8.0 s
Tiempo medidor entrada → medidor salida: 598.05 s
Tiempo total estimado: 10.37 min
Número de medidores en la ruta: 27
Ruta de medidores: [10527, 10528, 10639, 4245, 4257, 10036, 3849, 3846, 4410, 4406, 4409, 4433, 4456, 5696, 3583, 5695, 3425, 3424, 5703, 5691, 5624, 3834, 5625, 5692, 5622, 5627, 3421]


### PUERTA DEL SOL -> PLAZA CASTILLA

In [105]:
import pandas as pd
import numpy as np

df_validacion = pd.DataFrame([{
    "ruta_id": 1,
    "origen": "Puerta del Sol",
    "destino": "Plaza de Castilla",
    "id_medidor_entrada": 10527,
    "id_medidor_salida": 3421,
    "tiempo_origen_medidor_s": 16,
    "tiempo_grafo_medidores_s": 598.05,
    "tiempo_medidor_destino_s": 8,
    "tiempo_estimado_sistema_min": 10.37,
    "tiempo_real_referencia_min": np.nan,
    "fuente_referencia": "Google Maps"
}])

df_validacion

,ruta_id,origen,destino,id_medidor_entrada,id_medidor_salida,tiempo_origen_medidor_s,tiempo_grafo_medidores_s,tiempo_medidor_destino_s,tiempo_estimado_sistema_min,tiempo_real_referencia_min,fuente_referencia
0,1,Puerta del Sol,Plaza de Castilla,10527,3421,16,598.05,8,10.37,NaN,Google Maps


In [106]:
#Timpo estimado con google maps
df_validacion.loc[0, "tiempo_real_referencia_min"] = 39

In [107]:
#errores
df_validacion["error_min"] = (
    df_validacion["tiempo_estimado_sistema_min"]
    - df_validacion["tiempo_real_referencia_min"]
)

df_validacion["error_abs_min"] = df_validacion["error_min"].abs()

df_validacion["error_pct"] = (
    df_validacion["error_abs_min"]
    / df_validacion["tiempo_real_referencia_min"]
    * 100
)

df_validacion

,ruta_id,origen,destino,id_medidor_entrada,id_medidor_salida,tiempo_origen_medidor_s,tiempo_grafo_medidores_s,tiempo_medidor_destino_s,tiempo_estimado_sistema_min,tiempo_real_referencia_min,fuente_referencia,error_min,error_abs_min,error_pct
0,1,Puerta del Sol,Plaza de Castilla,10527,3421,16,598.05,8,10.37,39.0,Google Maps,-28.63,28.63,73.410256


In [108]:
metricas_validacion = pd.DataFrame({
    "num_rutas": [len(df_validacion)],
    "error_medio_absoluto_min": [df_validacion["error_abs_min"].mean()],
    "error_porcentual_medio_pct": [df_validacion["error_pct"].mean()],
    "sesgo_medio_min": [df_validacion["error_min"].mean()]
})

metricas_validacion

,num_rutas,error_medio_absoluto_min,error_porcentual_medio_pct,sesgo_medio_min
0,1,28.63,73.410256,-28.63


Puerta del Sol -> Plaza de Castilla

- Tiempo estimado por vuestro sistema: 10,37 min
- Tiempo real de referencia Google Maps: 39 min

- Error: -28,63 min
- Error absoluto: 28,63 min
- Error porcentual: 73,41 %